# Digital Cash Management Forecast: set up the project

This notebook sets up the Digital Cash Management Forecast in this Dataiku project (it also gives the project
that name), the way it runs in production. Everything the project writes is a CSV file, except `daily_flows` (Parquet
files in a folder, for reading speed).
It reads 2 warehouse tables, read-only, and creates or writes nothing in the warehouse: the clients
(`EDW.DIM_CUSTOMER`) and the daily postings (`EDW.FCT_STATEMENT_ENTRY`), which the staging recipe sums per client,
day and category in Dataiku. The first run reads each client's whole history; after that, each run reads only the
new days (through the database connection, a `SELECT` with a date filter) and adds them to `daily_flows`. It always
reads them through SQL, through the read-only database connection `EDW_CONNECTION`: at the bank, the warehouse;
on the test machine, a local test database with fake data (`warehouse_simulator.ipynb`). The same code, only the
connection differs. Names, codes and the categories (a mock list until the bank's transaction types are known):
`python/warehouse.py` in the project library.

It forecasts, per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 30, 60 and 90 days after the cut-off day (yesterday: days 1-30, 31-60, 61-90), as
P10, EV and P90. It builds 4 flow zones, each with an owner (and a 5th, Load test, apart):

- **Raw data** and **Staging** (shared): the 2 warehouse tables, read-only, then `clients` (the business clients)
  and `daily_flows` (money in and out per client, day and category): what every model gets. `staging_log` records
  each read.
- **Training** (data scientist): each training run is one model with its own **model ID**, trained with the
  **forecaster** the project variable `forecaster` names, tested on the last 6 periods of 30 days, tracked in Dataiku's
  **Experiment Tracking** and deployed as a new **version** of the saved model `cash_forecast`. A champion /
  challenger test activates the new version if it passes the **promotion rules** (project variables `promotion_*`).
- **Inference** (inference owner): `final_forecast`, the production model's forecast for every active client, made
  a chunk of clients at a time so it scales to the bank's ~420k clients (every row says which model and which load
  produced it), the forecast archive, and the inference log (one row per run, with timings and memory).
The data scientist tries new forecasters outside the flow, in the notebook `forecast_lab.ipynb` (it never touches
production), then sets the project variable `forecaster` and runs **Retrain, test and promote**.

- **Load test** (apart from everything else): the scenario **Load test** times the nightly forecast (no real
  training) on fake data of the same shape (clients, months, postings per month: set in its step 1) with a dummy model, and logs each run in
  `load_test_log`. It never reads or writes the real data, models or logs.

The zones only meet through three interfaces: **1. the data** (`daily_flows` and `clients`: staging → training, lab
and inference), **2. the forecaster** (`models.py`: lab → training), **3. the active version of the saved model**
(training → inference). Until the data scientist's models are ready, the forecaster is a placeholder,
`quantile_boosting`: 3 small models (EV, P10, P90), each shared by every category, with the category as an input. Two independent scenarios run it:
**Nightly batch** (inference) and **Retrain, test and promote** (training). Each starts by setting the cut-off day
(`set_cutoff`: yesterday, or the project variable `cutoff_override`) and checking the warehouse load of that day is
finished (`check_warehouse_loaded`: a mock for now, to replace with the bank's real check); the staging recipe stops
if the postings do not reach the cut-off day.

It then trains, tests and promotes a first model, and runs the first nightly batch.

**How to use it (at the bank):** in a new empty project, import this notebook only (Notebooks > New notebook >
Import), fill in the settings below (`EDW_CONNECTION`, `CODE_ENV`, ...) and run all cells (Cell > Run All). It first
checks it can read the 2 warehouse tables, and stops with a clear message if not. It can be run again in the same
project: what exists is kept (data, saved model versions, zones, scenario history), and the recipes' code is updated.

On the test machine only: run `warehouse_simulator.ipynb` first (it writes fake tables into the local test
database), then this notebook with `EDW_CONNECTION` set to that database's connection.

In [ ]:
PROJECT_NAME = "Digital Cash Management Forecast"

# ── 1. INPUT: the warehouse, read-only through SQL (nothing is ever written there) ──────────────────────────────────
# The project reads 2 tables of this schema: DIM_CUSTOMER and FCT_STATEMENT_ENTRY.
EDW_CONNECTION = "YOUR_EDW_CONNECTION"   # the Dataiku connection to the warehouse (Administration > Connections).
                                         # At the bank, for example "EDW_AI"; on the test machine "EDW_LOCAL"
EDW_SCHEMA = "EDW"                       # the schema that holds the 2 tables
HISTORY_MONTHS = 24                      # months of postings read the first time (and by the scenario Full rebuild),
                                         # up to yesterday. -1: the whole history. Each night then adds the new days
CLIENTS_LIMIT = -1                       # how many business clients to read (the lowest customer numbers): -1 all;
                                         # for example 1000 to test fast. Both: to change them later, run the setup
                                         # again, then the scenario Full rebuild

# ── 2. OUTPUT: where the project writes its own results ─────────────────────────────────────────────────────────────
OUTPUT_CONNECTION = "filesystem_managed"         # datasets (tables): clients, final_forecast, test_results, logs
OUTPUT_FOLDER_CONNECTION = "filesystem_folders"  # folders (files): daily_flows (Parquet), models, forecast archive

# ── 3. PYTHON ───────────────────────────────────────────────────────────────────────────────────────────────────────
CODE_ENV = "YOUR_CODE_ENV"               # an existing Python code env with scikit-learn and mlflow==2.22.0 (an admin
                                         # creates it once), for example "cashflow_forecast"

## 1. Code environment

The recipes need scikit-learn, and MLflow for Dataiku's Experiment Tracking, in an existing code env (`CODE_ENV`
above). This notebook does not create it: an admin creates it once. This checks it is there with those packages,
and otherwise stops with the request to send the admin.

In [ ]:
import dataiku

client = dataiku.api_client()
project = client.get_project(dataiku.default_project_key())
# scikit-learn: for the data scientist's models; mlflow: Experiment Tracking and the saved model. Dataiku tested MLflow models with
# mlflow 2.22.0 and 3.10.1 (DSS 14 and 15 docs, "MLflow models: limitations"); mlflow 3 needs Python 3.10 or later.
# Either works here; the request below asks for 2.22.0, which also runs on Python 3.9.
PACKAGES = ("scikit-learn", "mlflow==2.22.0")


def check_code_env(name, packages=PACKAGES):
    """Check the code env exists and lists the packages. It is not created here: an admin creates it once."""
    request = (f"Ask a Dataiku admin to create the Python code env {name} (Python 3.9 or later) with these "
               f"packages: {', '.join(packages)}. Or set CODE_ENV to an existing env that has them.")
    if name not in [e["envName"] for e in client.list_code_envs()]:
        raise RuntimeError(f"Code env {name} not found. {request}")
    try:
        listed = " ".join(client.get_code_env("PYTHON", name).get_settings().get_required_packages(as_list=True))
    except Exception:          # no right to read the env settings: it is checked when the first recipe runs
        print(f"Code env {name} found (could not read its packages)")
        return
    missing = [p for p in packages if p.split("=")[0].split(">")[0] not in listed]
    if missing:
        raise RuntimeError(f"Code env {name} lacks {', '.join(missing)}. {request}")
    print(f"Code env {name} has {', '.join(packages)}")


placeholders = [name for name, value in (("EDW_CONNECTION", EDW_CONNECTION), ("CODE_ENV", CODE_ENV))
                if str(value).startswith("YOUR_")]
if placeholders:
    raise RuntimeError(f"Fill in {' and '.join(placeholders)} in the settings cell above, then run again.")
check_code_env(CODE_ENV)

## 2. Code

The project library gets these files, by owner:

- shared: `warehouse.py` (the warehouse tables and columns read, codes, categories, daily sums, cut-off day, warehouse check) and
  `cashflow.py` (dates, the data interface, days to months, the saved model name)
- training and lab: `models.py` (the forecaster interface, the placeholder forecasters, the test and its scores) and
  `tracking.py` (model IDs, Experiment Tracking, saved model versions, promotion rules, champion / challenger)
- inference: `forecast.py` (the production model, the final forecast table)
- `forecasters.py`: the data scientist's own forecasters. Written only when it is created: running this notebook
  again never overwrites it.

Each recipe below becomes a Python recipe in the flow.

In [ ]:
WAREHOUSE_CODE = r'''
"""Warehouse tables (project library: python/warehouse.py).

The forecast reads 2 tables of the bank's warehouse, read-only, and creates or writes nothing there:
    DIM_CUSTOMER          EDW.DIM_CUSTOMER: the clients. The project keeps the business clients (companies and sole
                          proprietors) and only the columns below: personal fields are never read. Through the
                          database connection, a read-only SELECT of those columns (table_query), whole every run.
    FCT_STATEMENT_ENTRY   EDW.FCT_STATEMENT_ENTRY: the daily postings. The staging recipe sums them per client, day
                          and category (money in, money out) in Dataiku and adds the new days to daily_flows. Through
                          the database connection, the only query is a read-only SELECT of the columns below with a
                          date filter on VAL_DT (table_query): the last HISTORY_MONTHS months the first time (or after
                          the scenario Full rebuild), then only the days after the last day already in daily_flows.
Always through the database connection to the warehouse (the setup's EDW_CONNECTION: Oracle at the bank, read-only;
a local PostgreSQL with fake data on the test machine): the same code, only the connection differs. If a name or a
code differs at the bank, only this file changes.
"""
import numpy as np
import pandas as pd

# the tables the project reads. standard name: (table, where it is)
TABLES = {
    "clients": ("DIM_CUSTOMER", "EDW.DIM_CUSTOMER"),
    "transactions": ("FCT_STATEMENT_ENTRY", "EDW.FCT_STATEMENT_ENTRY"),
}
STAGED = ["clients", "daily_flows", "staging_log"]       # what the staging recipe writes

# Categories: each posting gets one from its transaction type (TRNS_TYP); a type not listed goes to OTHER.
# MOCK LIST: these are the sample data's types. At the bank, replace it with the real TRNS_TYP codes (ask the
# warehouse team for the list of codes and what each means); nothing else changes.
CATEGORY_OF_TYPE = {
    "COLLECTION": "CUSTOMER_RECEIPTS", "CARD_SETTLEMENT": "CUSTOMER_RECEIPTS", "CHEQUE": "CUSTOMER_RECEIPTS",
    "SALARY": "SALARIES", "RENT": "RENT", "SUPPLIER": "SUPPLIERS", "FINANCING": "FINANCING", "GOV": "GOVERNMENT",
    "UTILITY": "UTILITIES", "CARD": "CARDS", "FEE": "FEES",
}
OTHER = "OTHER"

# the columns read. standard name: warehouse (EDW) name
COLUMNS = {
    "clients": {"client_no": "CSTMR_CIF", "customer_type": "CSTMR_TYP", "line_of_business": "LOB_DSC",
                "legal_entity": "LGL_ENTY_DSC", "sub_legal_entity": "PRTY_SUB_LGL_DSC", "id_type": "PRTY_ID_TYP_DSC",
                "segment": "RM_SGMNT_NM", "sector": "SCTR_DSC", "business_sector": "BUS_SCTR_EN_NM",
                "industry": "INDST_EN_NM", "sub_industry": "SUB_INDST_EN_NM", "city": "CITY", "status": "STS_FLG"},
    # one row per posting. AMT_LCY: SAR, negative for money out (T24). TRNS_TYP: the transaction type, which gives
    # the category. RVRSL_MRKR: set on a reversing posting, which carries the reversed posting's TRNS_RFR
    "transactions": {"entry_id": "STMT_ENTRY_ID", "client_no": "CSTMR_CIF", "account_no": "ACNT_NMBR",
                     "value_date": "VAL_DT", "direction": "CRDT_DBT_FLG", "amount": "AMT_LCY", "txn_type": "TRNS_TYP",
                     "txn_ref": "TRNS_RFR", "reversal": "RVRSL_MRKR"},
}
# Which rows of DIM_CUSTOMER are read: 2 SQL conditions, applied in the warehouse (the bank's own query, checked on
# 2026-11-08: about 420k rows, one per customer). Column names in double quotes, so they work as they are on Oracle.
# The business clients: relationship manager code starting with 12, or the sector Sole Proprietorships
CLIENTS_BUSINESS = """ "PRIME_RM" LIKE '12%' OR "SCTR_DSC" = 'Sole Proprietorships' """
# DIM_CUSTOMER keeps history (about 520 million rows): each customer's current row is the one still open
CLIENTS_CURRENT_ROW = """ "END_DT" = DATE '2199-12-31' """
# FCT_STATEMENT_ENTRY is partitioned by month on its load date AS_OF_DT (not on the value date VAL_DT). Staging
# filters on it too, so Oracle reads only the load partitions that can hold the days it needs, not the whole table:
# AS_OF_DT from POSTINGS_LOAD_MARGIN_DAYS days before the first value day read. A posting loaded after its value
# date is always read; only one loaded more than that many days BEFORE its value date would be missed.
# None: no load-date filter (every partition is read).
POSTINGS_LOAD_DATE = "AS_OF_DT"
POSTINGS_LOAD_MARGIN_DAYS = 31
ACTIVE_STATUS = ("ACTIVE", "A")               # STS_FLG of an active client; an empty STS_FLG counts as active too
                                              # (an inactive one shows INACTIVE)
SOLE_PROPRIETOR = ("SOLE", "ESTABLISHMENT")   # words of LGL_ENTY_DSC, PRTY_SUB_LGL_DSC or SCTR_DSC for a business owned by
                                              # 1 person (the sector Sole Proprietorships, for example)


def read_columns(table):
    """The warehouse columns the project reads from a table."""
    return list(COLUMNS[table].values())


def to_standard(df, table):
    """A warehouse table with the standard column names and the project's values: client numbers as text, direction
    IN / OUT; for clients, status ACTIVE / CLOSED and sole_proprietor true / false. A missing column stops here
    with its name."""
    missing = [c for c in COLUMNS[table].values() if c not in df]
    if missing:
        raise ValueError(f"{TABLES[table][0]} has no column {', '.join(missing)}: check the warehouse table")
    out = pd.DataFrame({standard: df[column] for standard, column in COLUMNS[table].items()})
    out["client_no"] = out["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    if "direction" in out:
        out["direction"] = out["direction"].astype(str).str.strip().str.upper().str[0].map({"C": "IN", "D": "OUT"})
    if table == "clients":
        status_raw = out["status"].astype(str).str.strip().str.upper().replace(["NAN", "NONE", "NULL", ""], "")
        active = status_raw.isin(ACTIVE_STATUS) | status_raw.eq("")      # no status at all: active too
        out["status"] = np.where(active, "ACTIVE", "CLOSED")
        legal = (out["legal_entity"].fillna("").astype(str) + " " + out["sub_legal_entity"].fillna("").astype(str)
                 + " " + out["sector"].fillna("").astype(str))
        out["sole_proprietor"] = legal.str.upper().str.contains("|".join(SOLE_PROPRIETOR))
    return out


def one_row_per_client(clients):
    """One row per client number (the last one received). The warehouse already sends one per client
    (CLIENTS_CURRENT_ROW); staging warns if it did not."""
    return clients.drop_duplicates("client_no", keep="last").reset_index(drop=True)


def value_day(dates):
    """The day of each value date, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.normalize()


def table_query(table, name, since=None, end=None, count=False, limit=None):
    """The read-only query staging runs through the database connection: only the columns read (COLUMNS) of a table,
    named in the query, so it never depends on the column list Dataiku keeps for the dataset. name: the table's full
    quoted name. For the postings, only the value dates from since (None: the whole history) to before end; the value
    date comes back as text YYYY-MM-DD (TO_CHAR: Oracle and PostgreSQL), so no time zone can move it. count: the
    same rows counted (COUNT(*)). For the clients, only the business clients' current rows (clients_filter): the
    warehouse leaves the other customers and the old versions out, so far fewer rows travel."""
    date = COLUMNS["transactions"]["value_date"] if table == "transactions" else None
    columns = [f"TO_CHAR(\"{date}\", 'YYYY-MM-DD') AS \"{date}\"" if c == date else f'"{c}"'
               for c in COLUMNS[table].values()]
    query = f"SELECT {'COUNT(*) AS N' if count else ', '.join(columns)} FROM {name}"
    if table == "clients":
        query += f" WHERE {clients_filter()}"       # business clients' current rows only: the rest stays in Oracle
        if limit:                                   # only the first `limit` business clients (CLIENTS_LIMIT)
            query += f' ORDER BY "{COLUMNS["clients"]["client_no"]}" FETCH FIRST {int(limit)} ROWS ONLY'
    if date:
        where = [f"\"{date}\" < DATE '{end:%Y-%m-%d}'"]
        if since is not None:
            where.insert(0, f"\"{date}\" >= DATE '{since:%Y-%m-%d}'")
        query += f" WHERE {' AND '.join(where)}"
    return query


def clients_filter():
    """The SQL condition on DIM_CUSTOMER: the business clients (CLIENTS_BUSINESS), only each one's current row
    (CLIENTS_CURRENT_ROW). Used for the clients and, as a subquery, for the postings."""
    current = CLIENTS_CURRENT_ROW.strip()
    return f"({CLIENTS_BUSINESS.strip()})" + (f" AND ({current})" if current else "")


def _text(value):
    return "'" + str(value).replace("'", "''") + "'"


def daily_totals_query(postings, clients, since, end, limit=None):
    """The read-only query staging runs for the postings: the warehouse itself sums them into daily totals, so only
    one row per client, day and category comes back (columns client_no, day, category, money_in, money_out,
    postings), not every posting. postings, clients: the 2 tables' full quoted names. The rules (Oracle and
    PostgreSQL; NULL-safe, as Oracle treats '' as NULL):
    - 3 steps: the selected customers (DIM_CUSTOMER with clients_filter, one row each), the postings of the
      selected days (value dates from since, None: no lower bound, to before end), then the postings joined with
      the customers; with since, also only the load dates from
      POSTINGS_LOAD_MARGIN_DAYS before it (POSTINGS_LOAD_DATE: the partitions Oracle reads);
    - without reversed postings: a posting with a reversal marker (RVRSL_MRKR) carries the reversed posting's
      reference (TRNS_RFR), and every posting with that reference is left out;
    - without transfers between the client's own accounts: on the same client and reference, a credit and a debit
      that net to zero;
    - category from the transaction type (CATEGORY_OF_TYPE, else OTHER); money_in, money_out: the credits and the
      debits as positive SAR amounts (AMT_LCY); postings: how many.
    Window functions (one pass, no NOT IN / NOT EXISTS lookups) keep it fast on a big table."""
    c = COLUMNS["transactions"]
    dates = [f"\"{c['value_date']}\" < DATE '{end:%Y-%m-%d}'"]
    if since is not None:
        dates.insert(0, f"\"{c['value_date']}\" >= DATE '{since:%Y-%m-%d}'")
        if POSTINGS_LOAD_DATE and POSTINGS_LOAD_MARGIN_DAYS is not None:     # only the load partitions needed
            loaded = since - pd.Timedelta(days=POSTINGS_LOAD_MARGIN_DAYS)
            dates.append(f"\"{POSTINGS_LOAD_DATE}\" >= DATE '{loaded:%Y-%m-%d}'")
    cif = COLUMNS["clients"]["client_no"]
    # 1. the selected customers: one row per customer (DISTINCT), so the join below can never count a posting twice
    customers = f"SELECT DISTINCT \"{cif}\" AS c_cif FROM {clients} WHERE {clients_filter()}"
    if limit:                                       # the same first `limit` business clients as table_query
        customers += f" ORDER BY c_cif FETCH FIRST {int(limit)} ROWS ONLY"
    categories = " ".join(f"WHEN {_text(t)} THEN {_text(k)}" for t, k in CATEGORY_OF_TYPE.items())
    category = f"CASE UPPER(TRIM(p_typ)) {categories} ELSE {_text(OTHER)} END"
    return f"""WITH customers AS (
  {customers}
), postings AS (
  SELECT "{c['client_no']}" AS p_cif, TO_CHAR("{c['value_date']}", 'YYYY-MM-DD') AS p_day,
         UPPER(SUBSTR(TRIM("{c['direction']}"), 1, 1)) AS p_dir, "{c['amount']}" AS p_amt,
         "{c['txn_type']}" AS p_typ, "{c['txn_ref']}" AS p_ref, "{c['reversal']}" AS p_rev, "{c['entry_id']}" AS p_id
  FROM {postings}
  WHERE {' AND '.join(dates)}
), p AS (
  SELECT postings.* FROM postings JOIN customers ON customers.c_cif = postings.p_cif
), marked AS (
  SELECT p.*, MAX(CASE WHEN p_ref IS NOT NULL AND LENGTH(TRIM(p_rev)) > 0 AND TRIM(p_rev) NOT IN ('nan', 'None')
                       THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref) AS reversed
  FROM p
), kept AS (
  SELECT marked.*,
         MAX(CASE WHEN p_dir = 'C' THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref, p_cif)
         + MAX(CASE WHEN p_dir = 'D' THEN 1 ELSE 0 END) OVER (PARTITION BY p_ref, p_cif) AS sides,
         COALESCE(ABS(SUM(p_amt) OVER (PARTITION BY p_ref, p_cif)), 0) AS net
  FROM marked WHERE reversed = 0
)
SELECT p_cif AS "client_no", p_day AS "day", {category} AS "category",
       SUM(CASE WHEN p_dir = 'C' THEN ABS(p_amt) ELSE 0 END) AS "money_in",
       SUM(CASE WHEN p_dir = 'D' THEN ABS(p_amt) ELSE 0 END) AS "money_out",
       COUNT(p_id) AS "postings"
FROM kept
WHERE p_ref IS NULL OR NOT (sides = 2 AND net < 0.005)
GROUP BY p_cif, p_day, {category}"""


def standard_daily(df):
    """The daily totals as the warehouse returns them, with the project's types: client numbers as text, the day as
    a date, amounts and counts as numbers."""
    out = pd.DataFrame({"client_no": df["client_no"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True),
                        "day": pd.to_datetime(df["day"]),
                        "category": df["category"].astype(str),
                        "money_in": pd.to_numeric(df["money_in"]).astype(float).fillna(0.0),
                        "money_out": pd.to_numeric(df["money_out"]).astype(float).fillna(0.0),
                        "postings": pd.to_numeric(df["postings"]).astype("int64")})
    return out


def set_cutoff():
    """The forecast's cut-off: the last day of data, by default yesterday. The project variable cutoff_override
    (YYYY-MM-DD) fixes it instead: the sandbox's simulator sets it, and it reruns a past day. Sets the project
    variables cutoff_date, as_of_date and load_id (the data's as-of date), which the recipes read. The staging recipe
    stops if the warehouse has no posting up to the cut-off day.
    Run by the first step of each scenario: from warehouse import set_cutoff; set_cutoff()"""
    import dataiku
    project = dataiku.api_client().get_default_project()
    variables = project.get_variables()
    override = variables["standard"].get("cutoff_override")
    cutoff = pd.Timestamp(override) if override else pd.Timestamp.now().normalize() - pd.Timedelta(days=1)
    variables["standard"].update(cutoff_date=f"{cutoff:%Y-%m-%d}", as_of_date=f"{cutoff:%Y-%m-%d}",
                                 load_id=f"EDW_{cutoff:%Y%m%d}")
    project.set_variables(variables)
    print(f"Cut-off day {cutoff:%Y-%m-%d}" + (" (cutoff_override)" if override else " (yesterday)"))
    return f"{cutoff:%Y-%m-%d}"


def check_warehouse_loaded():
    """MOCK. Second step of each scenario: stops the scenario if the warehouse's load of the cut-off day is not
    finished (the ETL from the source systems into EDW). It always passes for now. Replace its body with the bank's
    real check, for example a read-only query on the warehouse's load-control table:
        SELECT STATUS FROM EDW.<LOAD_CONTROL_TABLE> WHERE LOAD_DATE = DATE '<cutoff_date>'   -- must say finished
    and raise an error if it is not finished. Whatever it says, staging also stops if the warehouse has no posting
    up to the cut-off day. Run as: from warehouse import check_warehouse_loaded; check_warehouse_loaded()"""
    import dataiku
    cutoff = dataiku.api_client().get_default_project().get_variables()["standard"]["cutoff_date"]
    print(f"MOCK warehouse check: the load of {cutoff} is assumed finished. Replace check_warehouse_loaded in "
          "warehouse.py with the bank's real check.")
    return True


def clear_daily_flows():
    """Step of the scenario Full rebuild (after the cut-off and the warehouse check): empties daily_flows and staging_log, so that the
    staging recipe reads the whole history again. Run as: from warehouse import clear_daily_flows; clear_daily_flows()"""
    import dataiku
    from cashflow import clear_daily
    clear_daily()
    dataiku.api_client().get_default_project().get_dataset("staging_log").clear()
    print("daily_flows and staging_log emptied: the staging recipe will read the whole history")


for _table, _columns in COLUMNS.items():          # warehouse names unique within a table
    assert len(set(_columns.values())) == len(_columns), _table
'''

In [ ]:
LIBRARY_CODE = r'''
"""Shared by every zone (project library: python/cashflow.py): dates, the business rule on amounts, and the
interfaces between the zones:
    1. the data (staging -> training, lab and inference): daily_flows (one row per client, day and category:
       money_in, money_out, postings, over each client's whole history; a folder of Parquet files, read with
       read_daily or iter_clients) and the dataset clients (one row per business client). Every model gets these
       and nothing else; period_matrix below turns the days into 30-day periods for the models that work that way.
    2. the forecaster (lab -> training): models.py.
    3. the saved model (training -> inference): SAVED_MODEL; inference uses its active version.
The forecast: per client, per category and in TOTAL (all categories together), the net cash flow (money in minus
money out) of each of the next 3 windows of 30 days after the cut-off day (the last day of data: yesterday): days
1-30, 31-60 and 61-90, as P10, EV and P90 in SAR."""
import numpy as np
import pandas as pd

PERIOD_DAYS = 30            # the forecast's windows, and the periods the models work with, are 30 days long
HORIZONS = (1, 2, 3)        # the windows forecast: 1 = days 1-30 after the cut-off day, 2 = days 31-60, 3 = 61-90
TOTAL = "TOTAL"             # the category of a client's all-categories net cash flow
SAVED_MODEL = "cash_forecast"
MIN_HISTORY_DAYS = 360      # a client needs this many days of history up to the cut-off to get a forecast (else
                            # status NOT_ENOUGH_HISTORY in the final forecast)
MIN_TRAINING_MONTHS = 18    # history needed to train and test a model: MIN_HISTORY_DAYS, plus the 6 periods of 30
                            # days the test keeps aside (about 540 days). Less is fine to test staging only
FORECAST_COLUMNS = ["client_no", "category", "window", "p10", "ev", "p90"]   # what a forecaster returns


def _plain(df):
    """Dates as plain (time zone free) timestamps, client numbers as text."""
    for c in df.columns:
        if isinstance(df[c].dtype, pd.DatetimeTZDtype):
            df[c] = df[c].dt.tz_convert(None)
    if "client_no" in df:
        df["client_no"] = df["client_no"].astype(str)
    return df


def read_dataset(name, columns=None):
    """A dataset as a DataFrame, with dates as plain (time zone free) timestamps and client numbers as text."""
    import dataiku
    return _plain(dataiku.Dataset(name).get_dataframe(columns=columns))


DAILY_FLOWS = "daily_flows"     # interface 1: a managed folder of Parquet files, written by staging
DAILY_COLUMNS = ["client_no", "day", "category", "money_in", "money_out", "postings"]
CURRENT = "/current.txt"        # in the folder: which version of the files is the current one


def empty_daily():
    """daily_flows with no row, with its column types."""
    return pd.DataFrame({"client_no": pd.Series(dtype=str), "day": pd.Series(dtype="datetime64[ns]"),
                         "category": pd.Series(dtype=str), "money_in": pd.Series(dtype=float),
                         "money_out": pd.Series(dtype=float), "postings": pd.Series(dtype="int64")})


def _daily_files(name=DAILY_FLOWS, **kwargs):
    """The folder and the current version's Parquet files, in order ([] when nothing is staged yet).
    kwargs: for dataiku.Folder (ignore_flow=True to read it in the recipe that writes it)."""
    import dataiku
    folder = dataiku.Folder(name, **kwargs)
    paths = folder.list_paths_in_partition()
    if CURRENT not in paths:
        return folder, []
    with folder.get_download_stream(CURRENT) as stream:
        version = stream.read().decode().strip()
    return folder, sorted(p for p in paths if p.startswith(f"/{version}/") and p.endswith(".parquet"))


def iter_daily_parts(name=DAILY_FLOWS, columns=None, **kwargs):
    """daily_flows one Parquet file at a time (DataFrames). Each file holds complete clients."""
    import io
    import pyarrow.parquet as pq
    folder, paths = _daily_files(name, **kwargs)
    for path in paths:
        with folder.get_download_stream(path) as stream:
            data = stream.read()
        yield _plain(pq.read_table(io.BytesIO(data), columns=columns).to_pandas())


def read_daily(name=DAILY_FLOWS, columns=None, **kwargs):
    """All of daily_flows as one DataFrame (for the sample data or a training set that fits in memory; for all the
    bank's clients, use iter_clients)."""
    parts = list(iter_daily_parts(name, columns, **kwargs))
    return pd.concat(parts, ignore_index=True) if parts else empty_daily()


def write_daily(parts, name=DAILY_FLOWS):
    """Write daily_flows again: each part (a DataFrame of complete clients, each client's rows together) becomes one
    Parquet file of a new version; only once all are written does the new version become the current one, and the
    older versions are removed. So a run that fails half way leaves the current version as it was. Returns the rows
    written."""
    import io
    import uuid
    import dataiku
    import pyarrow as pa
    import pyarrow.parquet as pq
    folder = dataiku.Folder(name)
    version = f"v{pd.Timestamp.now():%Y%m%dT%H%M%S}_{uuid.uuid4().hex[:6]}"
    rows = 0
    for i, part in enumerate(parts):
        buffer = io.BytesIO()
        pq.write_table(pa.Table.from_pandas(part[DAILY_COLUMNS], preserve_index=False), buffer)
        with folder.get_writer(f"/{version}/part_{i:05d}.parquet") as writer:
            writer.write(buffer.getvalue())
        rows += len(part)
    with folder.get_writer(CURRENT) as writer:
        writer.write(version.encode())
    for path in folder.list_paths_in_partition():
        if path != CURRENT and not path.startswith(f"/{version}/"):
            folder.delete_path(path)
    return rows


def clear_daily(name=DAILY_FLOWS):
    """Remove every file of daily_flows (the scenario Full rebuild)."""
    import dataiku
    folder = dataiku.Folder(name)
    for path in folder.list_paths_in_partition():
        folder.delete_path(path)


def iter_clients(name=DAILY_FLOWS, clients=5000, keep=None, **kwargs):
    """daily_flows read in chunks of `clients` COMPLETE clients: every row of a client is in the same chunk, so memory
    holds one chunk of clients, never the whole table. Staging keeps daily_flows grouped by client, so it is read
    straight through, one Parquet file at a time. keep: a function applied to each file's rows first (a filter).
    Stops with an error if a client's rows are not together (then run the scenario Full rebuild)."""
    done, pending, tail = set(), None, None

    def batch(out):
        ids = set(out["client_no"])
        if ids & done:
            raise ValueError(f"{name} is not grouped by client (client {next(iter(ids & done))} is in two places): "
                             "run the scenario Full rebuild of the daily flows")
        done.update(ids)
        return out.reset_index(drop=True)

    for chunk in iter_daily_parts(name, **kwargs):
        chunk = keep(chunk) if keep else chunk
        if tail is not None:
            chunk = pd.concat([tail, chunk], ignore_index=True)
        if chunk.empty:
            tail = None
            continue
        last = chunk["client_no"].iloc[-1]          # its rows may go on in the next chunk: kept for then
        at_end = chunk["client_no"].eq(last)
        tail = chunk[at_end]
        pending = chunk[~at_end] if pending is None else pd.concat([pending, chunk[~at_end]], ignore_index=True)
        ids = pending["client_no"].unique()
        while len(ids) >= clients:                  # every full chunk of clients
            first = pending["client_no"].isin(set(ids[:clients]))
            yield batch(pending[first])
            pending, ids = pending[~first], ids[clients:]
    rest = [f for f in (pending, tail) if f is not None and len(f)]
    if rest:
        rest = pd.concat(rest, ignore_index=True)
        ids = rest["client_no"].unique()
        for i in range(0, len(ids), clients):
            yield batch(rest[rest["client_no"].isin(set(ids[i:i + clients]))])


def append_row(name, columns, row):
    """Add one row to a log dataset (an appended output of the recipe) with fixed column types, so that rows of
    failed runs (with empty values) never change how the earlier rows are read. columns: {name: Dataiku type}, one of
    string, bigint, double, boolean."""
    import dataiku
    kinds = {"string": "string", "bigint": "Int64", "double": "float64", "boolean": "boolean"}
    frame = pd.DataFrame([{c: row.get(c) for c in columns}])
    for c, kind in columns.items():
        frame[c] = (frame[c].fillna("").astype(str) if kind == "string"
                    else pd.to_numeric(frame[c]).astype(kinds[kind]) if kind != "boolean"
                    else frame[c].astype("boolean"))
    dataset = dataiku.Dataset(name)
    dataset.write_schema([{"name": c, "type": kind} for c, kind in columns.items()])
    with dataset.get_writer() as writer:
        writer.write_dataframe(frame)


class Progress:
    """Progress lines in the job's log, for long steps: how much is done (of how much, when known), the percentage,
    the time so far and about how long is left. One line at most every `every` seconds, and the last one always. They
    start with PROGRESS, so the setup notebook can show them while it waits for the job.
        progress = Progress("reading postings", total=41_000_000); progress.add(200_000) ... progress.finish()"""

    def __init__(self, what, total=None, every=30):
        import time
        self.what, self.total, self.every = what, total, every
        self.started = self.printed = time.time()
        self.done = 0
        print(f"PROGRESS {what}: starting" + (f", {total:,} to do" if total else ""), flush=True)

    def add(self, n=1):
        import time
        self.done += n
        if time.time() - self.printed >= self.every:
            self.show()

    def finish(self):
        self.show(last=True)

    def show(self, last=False):
        import time
        now = time.time()
        self.printed = now
        minutes = (now - self.started) / 60
        if last:
            print(f"PROGRESS {self.what}: done, {self.done:,} in {minutes:.1f} min", flush=True)
            return
        line = f"PROGRESS {self.what}: {self.done:,}"
        if self.total:
            share = min(self.done / self.total, 1.0)
            line += f" of {self.total:,} ({share:.0%})"
        line += f", {minutes:.1f} min so far"
        if self.total and 0 < share < 1:
            line += f", about {minutes * (1 - share) / share:.0f} min left"
        print(line, flush=True)


def waiting(what, every=60):
    """While the warehouse runs a query and no row has come back yet, a PROGRESS line every `every` seconds, so a
    long query never looks stuck. Returns an Event: set it when the first row arrives (or the query ends)."""
    import threading
    import time
    stop, started = threading.Event(), time.time()

    def beat():
        while not stop.wait(every):
            print(f"PROGRESS {what}: no row yet, the warehouse is still running the query "
                  f"({(time.time() - started) / 60:.0f} min)", flush=True)

    threading.Thread(target=beat, daemon=True).start()
    return stop


class Spread:
    """Rows spread over n files on local disk by the hash of a column, then read back one file at a time: all the
    rows with the same value of that column end up in the same file, and memory holds one file, not the whole table.
    Used by staging (postings by reference, then daily flows by client)."""

    def __init__(self, folder, n):
        import os
        os.makedirs(folder, exist_ok=True)
        self.n = max(1, int(n))
        self.paths = [os.path.join(folder, f"part_{i}.pkl") for i in range(self.n)]

    def add(self, df, key):
        """key: a column name, or a Series (same index as df) to spread by."""
        import pickle
        key = df[key] if isinstance(key, str) else key
        part = pd.util.hash_pandas_object(key.astype(str), index=False).to_numpy() % self.n
        for i, rows in df.groupby(part):
            with open(self.paths[i], "ab") as f:
                pickle.dump(rows, f, protocol=4)

    def parts(self):
        """Each file's rows (a DataFrame); files that got no rows are skipped."""
        import os
        import pickle
        for path in self.paths:
            if not os.path.exists(path):
                continue
            frames = []
            with open(path, "rb") as f:
                while True:
                    try:
                        frames.append(pickle.load(f))
                    except EOFError:
                        break
            yield pd.concat(frames, ignore_index=True)


def month_start(dates):
    """First day of the month, as a plain (time zone free) timestamp."""
    d = pd.to_datetime(pd.Series(dates))
    if getattr(d.dt, "tz", None) is not None:
        d = d.dt.tz_convert(None)
    return d.dt.to_period("M").dt.to_timestamp()


def cutoff_date():
    """The forecast's cut-off: the last day of data, by default yesterday (project variable cutoff_date, set by the
    first step of each scenario: warehouse.set_cutoff)."""
    import dataiku
    return pd.Timestamp(dataiku.get_custom_variables()["cutoff_date"])


def whole_number(name, value):
    """A setting that is a whole number, or None for "all" (empty, None, -1 or "all"). Stops with a clear message
    otherwise."""
    value = "" if value is None else str(value).strip()
    if value.lower() in ("", "none", "all", "-1"):
        return None
    try:
        number = float(value)
    except ValueError:
        number = 0.5
    if number != int(number) or number < 1:
        raise ValueError(f"{name} is {value!r}: it must be a whole number (1 or more), or -1 / None for all")
    return int(number)


def history_start(cutoff):
    """The first day staging reads when it reads everything (the first run, or the scenario Full rebuild): the
    project variable history_months months before the cut-off day, or None for the whole history. With less than
    about 18 months, few clients reach the history a forecast and the model test need (MIN_HISTORY_DAYS, and the
    last 6 periods of 30 days kept aside): fine to test the pipeline, not to judge a model."""
    import dataiku
    months = whole_number("history_months", dataiku.get_custom_variables().get("history_months"))
    return None if months is None else cutoff - pd.DateOffset(months=months) + pd.Timedelta(days=1)


def clients_limit():
    """How many business clients staging reads (project variable clients_limit): None for all of them."""
    import dataiku
    return whole_number("clients_limit", dataiku.get_custom_variables().get("clients_limit"))


def until(daily, cutoff):
    """The daily flows known at the cut-off: the days up to the cut-off day (included)."""
    return daily[pd.to_datetime(daily["day"]) <= cutoff]


def window(cutoff, h):
    """The first and last day of window h after the cut-off day (h = 1: days 1-30)."""
    return cutoff + pd.Timedelta(days=PERIOD_DAYS * (h - 1) + 1), cutoff + pd.Timedelta(days=PERIOD_DAYS * h)


def finish(ev, p10, p90, index=None):
    """P10 <= EV <= P90 (SAR). The net cash flow can be below zero."""
    ev, p10, p90 = (np.asarray(v, dtype=float) for v in (ev, p10, p90))
    return pd.DataFrame({"p10": np.minimum(p10, ev), "ev": ev, "p90": np.maximum(p90, ev)}, index=index).round(2)


def _sums(d, key):
    """Net cash flow per client, category (and TOTAL) and key."""
    by_category = d.groupby(["client_no", "category", key])["net"].sum()
    total = d.groupby(["client_no", key])["net"].sum().to_frame().assign(category=TOTAL)
    total = total.set_index("category", append=True).reorder_levels(["client_no", "category", key])["net"]
    return pd.concat([by_category, total])


def _wide(sums, columns):
    """One row per client and category, the given columns; 0 after a series' first value, empty before it."""
    if sums.empty:
        return pd.DataFrame(index=pd.MultiIndex.from_tuples([], names=["client_no", "category"]))
    wide = sums.unstack(sums.index.names[-1]).reindex(columns=columns)
    started = wide.notna().cumsum(axis=1) > 0
    return wide.fillna(0.0).where(started).sort_index()


def period_matrix(daily, cutoff):
    """The net cash flow (money in minus money out) per 30-day period ending at the cut-off day: one row per client
    and category (and per client in TOTAL), one column per period, named by its last day (the last column is the 30
    days up to the cut-off, the one before the 30 days before, ...). 0 for a period with no posting after the
    series' first period, empty before it. The window forecast h is the period ending cutoff + 30 x h days."""
    d = until(daily, cutoff)
    back = (cutoff - pd.to_datetime(d["day"])).dt.days // PERIOD_DAYS
    d = d.assign(period=(cutoff - pd.to_timedelta(back * PERIOD_DAYS, unit="D")).values,
                 net=(d["money_in"] - d["money_out"]).values)
    if d.empty:
        return _wide(pd.Series(dtype=float), [])
    oldest = int(back.max())
    return _wide(_sums(d, "period"), [cutoff - pd.Timedelta(days=PERIOD_DAYS * k) for k in range(oldest, -1, -1)])


def period_series(daily, cutoff):
    """period_matrix as rows: client_no, category, period (its last day), net (no row before a series' first)."""
    long = period_matrix(daily, cutoff).reset_index().melt(id_vars=["client_no", "category"], var_name="period",
                                                           value_name="net")
    long["period"] = pd.to_datetime(long["period"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "period"]).reset_index(drop=True)


def monthly_series(daily, cutoff):
    """For the EDA: the net cash flow per client, category (and TOTAL) and calendar month, up to the cut-off day (its
    month is partial): client_no, category, month, net (no row before a series' first month)."""
    d = until(daily, cutoff)
    d = d.assign(month=month_start(d["day"]).values, net=(d["money_in"] - d["money_out"]).values)
    if d.empty:
        return pd.DataFrame(columns=["client_no", "category", "month", "net"])
    months = list(pd.date_range(d["month"].min(), month_start([cutoff])[0], freq="MS"))
    long = _wide(_sums(d, "month"), months).reset_index().melt(id_vars=["client_no", "category"], var_name="month",
                                                                value_name="net")
    long["month"] = pd.to_datetime(long["month"])
    return long.dropna(subset=["net"]).sort_values(["client_no", "category", "month"]).reset_index(drop=True)


def history_days(daily, cutoff):
    """Per client: the days from its first posting to the cut-off day (both counted)."""
    first = pd.to_datetime(until(daily, cutoff).groupby("client_no")["day"].min())
    return ((cutoff - first).dt.days + 1).rename("history_days")
'''

In [ ]:
MODELS_CODE = r'''
"""Models (project library: python/models.py): interface 2, the forecaster, the placeholder forecasters and the test
every model goes through. Training (train_and_test, champion_challenger) and the lab use it; inference only calls the
production model's predict.

A forecaster is a class with:
    name            its name. The project variable forecaster says which one the scenario Retrain, test and promote
                    trains (default: quantile_boosting).
    params          its settings (a dict), recorded in Experiment Tracking
    fit(daily, clients, cutoff)
                    learns from the data known at the cut-off day (interface 1, cashflow.py): daily: daily_flows
                    (client_no, day, category, money_in, money_out, postings), every day up to the cut-off day;
                    clients: the clients dataset; cutoff: the cut-off day (a Timestamp: the last day of data).
                    cashflow.period_matrix turns the days into 30-day periods ending at the cut-off day.
    predict(daily, clients, cutoff)
                    the same inputs, for the clients to forecast. Returns a DataFrame with one row per client,
                    category (each category the client has in daily, and TOTAL: all its categories together) and
                    window (1 = days 1-30 after the cut-off day, 2 = days 31-60, 3 = days 61-90): the columns
                    client_no, category, window, p10, ev, p90 (the net cash flow of that window, SAR). forecast()
                    checks it and applies P10 <= EV <= P90.
The data scientist's own forecasters go in forecasters.py: the setup notebook creates that file once and never
overwrites it. lab_compare (notebook forecast_lab) tries forecasters against the production model without touching
production.
"""
import numpy as np
import pandas as pd

from cashflow import (FORECAST_COLUMNS, HORIZONS, PERIOD_DAYS, TOTAL, cutoff_date, finish, period_matrix,
                      period_series, until)

HOLDOUT = 6                 # the last 6 periods of 30 days are kept aside to test the models
OUTPUT = ["p10", "ev", "p90"]


class RecentAverage:
    """A simple forecaster with no machine learning, to compare others with (and fall back on). Per client and
    category, EV = the average net cash flow of the last 3 periods of 30 days (the same for the 3 windows). P10 and
    P90 = EV -/+ width x the series' standard deviation over the last 12 periods, where fit sets width so that, over
    the 12 periods before the cut-off, 80% of the actual periods fell inside."""
    name = "recent_average"
    params = {"periods": 3, "spread_periods": 12, "coverage": 0.8}

    def fit(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        n, s = self.params["periods"], self.params["spread_periods"]
        z = []
        for k in range(1, min(12, wide.shape[1] - 1) + 1):         # each of the last 12 periods, from the ones before
            past, actual = wide.iloc[:, :-k], wide.iloc[:, -k]
            ev, sd = past.iloc[:, -n:].mean(axis=1), past.iloc[:, -s:].std(axis=1)
            ok = actual.notna() & ev.notna() & (sd > 0)
            z.append(((actual - ev).abs() / sd)[ok])
        z = pd.concat(z) if z else pd.Series(dtype=float)
        self.width = float(z.quantile(self.params["coverage"])) if len(z) else 1.28
        return self

    def predict(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        if wide.empty:
            return pd.DataFrame(columns=FORECAST_COLUMNS)
        ev = wide.iloc[:, -self.params["periods"]:].mean(axis=1).fillna(0.0)
        sd = wide.iloc[:, -self.params["spread_periods"]:].std(axis=1).fillna(ev.abs())
        one = pd.DataFrame({"p10": ev - self.width * sd, "ev": ev, "p90": ev + self.width * sd}).reset_index()
        return pd.concat([one.assign(window=h) for h in HORIZONS], ignore_index=True)[FORECAST_COLUMNS]


class QuantileBoosting:
    """The production placeholder, until the data scientist's models replace it: 3 small models, one for the EV, one
    for P10 and one for P90 (scikit-learn gradient boosting; P10 and P90 with the quantile loss, the EV with the median,
    which is far less thrown off by a few huge periods than the mean). Each of the 3 is ONE model for every category
    and TOTAL: the category is one of its inputs, so a category with little data learns from the others.
    Inputs per client and category at a cut-off day: the net cash flow of the last 12 periods of 30 days divided by
    the series' scale (so small and large clients look alike), the category, how many periods are known, the window
    and the calendar month of the window's middle (seasonality). Trained on the cut-offs of the last 24 periods, on
    at most max_rows rows (a random sample beyond that, the same every run). Each row is forecast from its own
    client's history only."""
    name = "quantile_boosting"
    params = {"lags": 12, "cutoffs": 24, "max_iter": 200, "learning_rate": 0.05, "max_rows": 1_000_000}
    QUANTILES = {"ev": 0.5, "p10": 0.1, "p90": 0.9}

    def _rows(self, wide, p, h):
        """Inputs at period position p (the cut-off) for window h, and each series' scale."""
        last = wide.iloc[:, max(0, p - self.params["lags"] + 1):p + 1]
        scale = last.abs().mean(axis=1).fillna(0).clip(lower=100.0)
        X = last.div(scale, axis=0).fillna(0.0)
        X.columns = [f"lag_{len(X.columns) - i}" for i in range(len(X.columns))]
        X = X.reindex(columns=[f"lag_{k}" for k in range(self.params["lags"], 0, -1)], fill_value=0.0)
        category = pd.Categorical(wide.index.get_level_values("category"), categories=self.categories)
        X["category"] = np.where(category.codes >= 0, category.codes, np.nan)      # a new category: unknown
        X["periods_known"] = last.notna().sum(axis=1).values
        X["window"] = h
        X["calendar_month"] = (wide.columns[p] + pd.Timedelta(days=PERIOD_DAYS * h - PERIOD_DAYS // 2)).month
        return X, scale

    def fit(self, daily, clients, cutoff):
        return self.fit_periods(period_matrix(daily, cutoff))

    def predict(self, daily, clients, cutoff):
        return self.predict_periods(period_matrix(daily, cutoff))

    def fit_periods(self, wide):
        """fit, from the net cash flow per 30-day period (cashflow.period_matrix)."""
        from sklearn.ensemble import HistGradientBoostingRegressor
        self.categories = sorted(set(wide.index.get_level_values("category")))
        n = wide.shape[1]
        blocks = [(p, h) for p in range(max(0, n - 1 - self.params["cutoffs"]), n - 1) for h in HORIZONS if p + h < n]
        keep = min(1.0, self.params["max_rows"] / max(1, len(wide) * len(blocks)))
        rng, Xs, ys = np.random.default_rng(0), [], []
        for p, h in blocks:
            X, scale = self._rows(wide, p, h)
            y = wide.iloc[:, p + h] / scale
            ok = (y.notna() & wide.iloc[:, p].notna()).values & (rng.random(len(y)) < keep)
            Xs.append(X[ok])
            ys.append(y[ok])
        X, y = pd.concat(Xs), pd.concat(ys)
        settings = {"max_iter": self.params["max_iter"], "learning_rate": self.params["learning_rate"],
                    "categorical_features": ["category"], "random_state": 0}
        self.models = {column: HistGradientBoostingRegressor(loss="quantile", quantile=q, **settings).fit(X, y)
                       for column, q in self.QUANTILES.items()}
        return self

    def predict_periods(self, wide):
        """predict, from the net cash flow per 30-day period (cashflow.period_matrix)."""
        if wide.empty:
            return pd.DataFrame(columns=FORECAST_COLUMNS)
        out = []
        for h in HORIZONS:
            X, scale = self._rows(wide, wide.shape[1] - 1, h)
            rows = wide.index.to_frame(index=False).assign(window=h)
            for column, model in self.models.items():
                rows[column] = model.predict(X) * scale.values
            out.append(rows)
        return pd.concat(out, ignore_index=True)[FORECAST_COLUMNS]


def known_forecasters():
    """The forecasters Retrain can train, by name: the placeholders (quantile_boosting, recent_average) and those
    listed in FORECASTERS in forecasters.py. Also returns the problem reading forecasters.py, if any (the placeholders
    still work)."""
    found, problem = {QuantileBoosting.name: QuantileBoosting, RecentAverage.name: RecentAverage}, None
    try:
        import forecasters
        for forecaster in forecasters.FORECASTERS:
            found[forecaster.name] = forecaster
    except Exception as e:
        problem = f"forecasters.py could not be read ({type(e).__name__}: {e})"
    return found, problem


def new_forecaster(name=None):
    """A new, untrained forecaster: the one named by the project variable forecaster (default quantile_boosting)."""
    import dataiku
    name = name or dataiku.get_custom_variables().get("forecaster") or QuantileBoosting.name
    found, problem = known_forecasters()
    if name not in found:
        raise ValueError(f"No forecaster named {name!r} (project variable forecaster). Known: {', '.join(found)}. "
                         + (problem if problem else "Add yours to FORECASTERS in forecasters.py."))
    forecaster = found[name]()
    missing = [a for a in ("name", "fit", "predict") if not hasattr(forecaster, a)]
    if missing:
        raise ValueError(f"Forecaster {name} has no {', '.join(missing)} (see models.py)")
    return forecaster


def forecast(forecaster, daily, clients, cutoff):
    """P10, EV and P90 of a trained forecaster (SAR), checked, with P10 <= EV <= P90: one row per client, category and
    window. Everything that predicts goes through here: the test, champion / challenger, the lab and the
    nightly forecast."""
    out = forecaster.predict(daily, clients, cutoff)
    name = getattr(forecaster, "name", type(forecaster).__name__)
    if not isinstance(out, pd.DataFrame) or not set(FORECAST_COLUMNS) <= set(out.columns):
        raise ValueError(f"Forecaster {name}: predict must return a DataFrame with the columns "
                         f"{', '.join(FORECAST_COLUMNS)}")
    empty = [c for c in OUTPUT if out[c].isna().any()]
    if empty:
        raise ValueError(f"Forecaster {name}: predict returned empty values in {', '.join(empty)}")
    out = out[FORECAST_COLUMNS].reset_index(drop=True)
    out["client_no"] = out["client_no"].astype(str)
    out["window"] = out["window"].astype(int)
    out[OUTPUT] = finish(out["ev"], out["p10"], out["p90"], out.index)
    return out.sort_values(["client_no", "category", "window"]).reset_index(drop=True)


def active(daily, clients):
    """The active clients and their daily flows: the clients the test and the forecast are made for."""
    clients = clients[clients["status"] == "ACTIVE"]
    return daily[daily["client_no"].isin(set(clients["client_no"]))], clients


def test_forecasts(fitted, daily, clients, cutoff):
    """The test, the same for every model: a forecaster trained without the last 6 periods of 30 days forecasts the
    windows after each test cut-off (the 6 days, 30 days apart, before the cut-off) from the data known at that day,
    for the active clients. Returns one row per client, category, test cut-off and window whose days are known:
    target (the actual net cash flow), p10, ev, p90 and baseline (the simple guess: the same as the 30 days before the
    test cut-off)."""
    daily, clients = active(daily, clients)
    known = period_series(daily, cutoff)
    actual = known.rename(columns={"period": "target_end", "net": "target"})
    last = known.rename(columns={"period": "test_cutoff", "net": "baseline"})
    parts = []
    for k in range(HOLDOUT):
        t = cutoff - pd.Timedelta(days=PERIOD_DAYS * (HOLDOUT - k))
        f = forecast(fitted, until(daily, t), clients, t).assign(test_cutoff=t)
        f["target_end"] = f["window"].map({h: t + pd.Timedelta(days=PERIOD_DAYS * h) for h in HORIZONS})
        parts.append(f[f["target_end"] <= cutoff])
    keys = ["client_no", "category"]
    tested = (pd.concat(parts, ignore_index=True)
              .merge(actual, on=keys + ["target_end"], how="left")
              .merge(last, on=keys + ["test_cutoff"], how="left"))
    tested[["target", "baseline"]] = tested[["target", "baseline"]].fillna(0.0)
    return tested


def backtest(forecaster, daily, clients, cutoff):
    """Train the forecaster without the last 6 periods of 30 days (the data known at the first test cut-off), then
    test it on them (test_forecasts). Returns the test rows; the forecaster is now the backtest forecaster."""
    first = cutoff - pd.Timedelta(days=PERIOD_DAYS * HOLDOUT)
    forecaster.fit(until(daily, first), clients, first)
    return test_forecasts(forecaster, daily, clients, cutoff)


def scores(d):
    """Test scores of predictions (columns target, p10, ev, p90, baseline): EV error (WAPE) of the model and of the
    simple guess, and the share of actual values below P10, between P10 and P90, and above P90."""
    size = d["target"].abs().sum()
    wape = lambda column: round((d["target"] - d[column]).abs().sum() / size, 4) if size else np.nan
    return pd.Series({"rows": len(d), "actual_total": round(d["target"].sum(), 0), "wape_model": wape("ev"),
                      "wape_baseline": wape("baseline") if "baseline" in d else np.nan,
                      "coverage_p10_p90": round(((d["target"] >= d["p10"]) & (d["target"] <= d["p90"])).mean(), 4),
                      "below_p10": round((d["target"] < d["p10"]).mean(), 4),
                      "above_p90": round((d["target"] > d["p90"]).mean(), 4)})


def test_results(tested):
    """Test scores per window (1, 2, 3) and overall (window ALL), for the TOTAL rows (level TOTAL) and the
    category rows (level CATEGORIES). The overall scores, which promotion uses, are on all rows."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    level = np.where(tested["category"] == "TOTAL", "TOTAL", "CATEGORIES")
    results = pd.concat([
        tested.assign(level=level).groupby(["level", "window"])[scored].apply(scores).reset_index(),
        tested.assign(level=level).groupby("level")[scored].apply(scores).reset_index().assign(window="ALL"),
        scores(tested).to_frame().T.assign(level="ALL", window="ALL")], ignore_index=True)
    results["window"] = results["window"].astype(str)
    return results[["level", "window"] + [c for c in results if c not in ("level", "window")]]


def run_metrics(results):
    """The scores recorded in Experiment Tracking: overall, and the EV error of the TOTAL rows per window.
    results: from test_results."""
    overall = results[(results["level"] == "ALL")].iloc[0]
    metrics = {k: overall[c] for k, c in (("wape", "wape_model"), ("wape_baseline", "wape_baseline"),
                                          ("coverage_p10_p90", "coverage_p10_p90"), ("below_p10", "below_p10"),
                                          ("above_p90", "above_p90"))}
    for row in results[(results["level"] == "TOTAL") & (results["window"] != "ALL")].itertuples():
        metrics[f"wape_total_w{row.window}"] = row.wape_model
    return metrics


def _lab_test():
    """The lab's data and the production model's test, the same as champion / challenger's: daily flows, clients,
    the production model's ID and its backtest forecaster's test rows (None without a production model)."""
    import tracking
    from cashflow import read_daily, read_dataset
    daily, clients = read_daily(), read_dataset("clients")
    production_id = tracking.production_model_id()
    production = (test_forecasts(tracking.load_backtest(production_id), daily, clients, cutoff_date())
                  if production_id else None)
    return daily, clients, production_id, production


def _by_window(tested):
    """Overall test scores per window (all rows), indexed by window (1, 2, 3, ALL)."""
    scored = ["target", "p10", "ev", "p90", "baseline"]
    per = tested.groupby("window")[scored].apply(scores)
    per.index = per.index.astype(str)
    per.loc["ALL"] = scores(tested)
    return per


def _record_try(forecaster, tested, production_id):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab). Returns the run ID."""
    import tracking
    params = {"forecaster": forecaster.name, **getattr(forecaster, "params", {}),
              "cutoff_date": f"{cutoff_date():%Y-%m-%d}", "test_rows": len(tested),
              "compared_with": production_id or "none"}
    run_name = f"{forecaster.name}_{pd.Timestamp.now():%Y%m%dT%H%M}"
    return tracking.log_lab_run(run_name, params, run_metrics(test_results(tested)), forecaster)


def lab_compare(forecasters, record=True):
    """Try several forecasters at once against the production model, on the same test as champion / challenger,
    without touching production. Each forecaster is trained without the last 6 periods of 30 days; each try is
    recorded in Experiment Tracking (experiment cashflow_lab) unless record is False. Returns a dict:
        board       one row per model (production first, simple guess last): EV error, P10-P90 coverage, share below
                    P10 and above P90, the decision champion / challenger would take and its reason, the lab run ID
        errors      EV error per window: the simple guess, the production model and each forecaster
        coverage    P10-P90 coverage per window: the production model and each forecaster
        tested      the test rows with each model's p10, ev, p90, by name, for your own analysis"""
    import tracking
    names = [f.name for f in forecasters]
    if len(set(names)) != len(names) or {"production", "simple_guess_baseline"} & set(names):
        raise ValueError(f"Give each forecaster its own name: {names}")
    daily, clients, production_id, production = _lab_test()
    tested = {} if production is None else {"production": production}
    for forecaster in forecasters:
        tested[forecaster.name] = backtest(forecaster, daily, clients, cutoff_date())
    run_ids = {f.name: _record_try(f, tested[f.name], production_id) for f in forecasters} if record else {}
    reference = scores(production) if production is not None else None
    board = []
    for name, rows in tested.items():
        s = scores(rows)
        decision, reason = (("", f"the production model, {production_id}") if name == "production"
                            else tracking.decide(s, reference))
        board.append({"model": name, "ev_error": s["wape_model"], "coverage_p10_p90": s["coverage_p10_p90"],
                      "below_p10": s["below_p10"], "above_p90": s["above_p90"], "decision": decision,
                      "reason": reason, "lab_run_id": run_ids.get(name, "")})
    board.append({"model": "simple_guess_baseline", "ev_error": scores(next(iter(tested.values())))["wape_baseline"],
                  "decision": "", "reason": "the same as the 30 days before: the bar to beat", "lab_run_id": ""})
    board = pd.DataFrame(board).set_index("model")
    per = {name: _by_window(rows) for name, rows in tested.items()}
    first = next(iter(per.values()))
    errors = pd.DataFrame({"simple_guess_baseline": first["wape_baseline"],
                           **{name: found["wape_model"] for name, found in per.items()}})
    coverage = pd.DataFrame({name: found["coverage_p10_p90"] for name, found in per.items()})
    return {"board": board, "errors": errors, "coverage": coverage, "tested": tested}
'''

In [ ]:
TRACKING_CODE = r'''
"""Model tracking and promotion (project library: python/tracking.py), for training and the lab.

Every training run is one model with its own model ID, recorded in Dataiku's Experiment Tracking (MLflow; artifacts in
the managed folder "experiments"): its forecaster and settings, the cut-off, test scores on the last 6 periods of
30 days, the backtest forecaster (trained without those periods, for champion / challenger tests) and one MLflow model (the final
forecaster, which returns P10, EV and P90). It is deployed as a new version (version ID = model ID) of the saved model
cash_forecast (interface 3, cashflow.SAVED_MODEL). The production model is the active version; promoting a model
activates it, and the nightly forecast always uses the active version.
Lab tries (notebook forecast_lab) go to their own experiment, cashflow_lab, in the same folder. They are never
deployed.
"""
import inspect
import os
import pickle
import sys
import tempfile

import pandas as pd

from cashflow import SAVED_MODEL

EXPERIMENT, FOLDER = "cashflow_forecast", "experiments"
LAB_EXPERIMENT, LAB_FOLDER = "cashflow_lab", FOLDER        # lab tries: their own experiment, same folder
# The promotion rules: project variables (change them in the project's variables, no code change), with these
# defaults. A candidate is promoted only if all hold:
PROMOTION_RULES = {
    "promotion_min_coverage": 0.70,      # its P10-P90 coverage is at least this...
    "promotion_max_coverage": 0.90,      # ...and at most this (80% is the aim)
    "promotion_beat_simple_guess": True,  # its EV error is lower than the simple guess's (same as the month before)
    "promotion_min_gain": 0.0,           # its EV error is lower than the production model's by more than this share
                                         # (0.02: at least 2% lower). A tie keeps the production model.
}


def _project():
    import dataiku
    return dataiku.api_client().get_default_project()


def _mlflow(folder):
    """Experiment Tracking in the folder. By its ID: outside a recipe (the lab notebook), setup_mlflow does not resolve
    a folder name."""
    import dataiku
    return _project().setup_mlflow(managed_folder=dataiku.Folder(folder).get_id())


def _experiment_name(mlflow, experiment, folder):
    """The experiment, kept in the folder. If an experiment of that name points to another folder (the folder was
    deleted and created again, for example when the flow was deleted by hand), its files are gone: use
    <experiment>_<folder id> instead."""
    import dataiku
    folder_id = dataiku.Folder(folder).get_id()
    found = mlflow.get_experiment_by_name(experiment)
    if found is None or f"//{folder_id}/" in (found.artifact_location or ""):
        return experiment
    return f"{experiment}_{folder_id}"


def new_model_id(cutoff):
    """CFF_<cut-off day>_<training time>, for example CFF_20261031_20261104T1215 (also the saved model version)."""
    return f"CFF_{cutoff:%Y%m%d}_{pd.Timestamp.now():%Y%m%dT%H%M}"


def promotion_rules():
    """The promotion rules: the project variables PROMOTION_RULES names, or their defaults."""
    import dataiku
    variables = dataiku.get_custom_variables()
    rules = {}
    for name, default in PROMOTION_RULES.items():
        value = variables.get(name, default)
        rules[name] = (str(value).strip().lower() in ("true", "1", "yes")) if isinstance(default, bool) else float(value)
    return rules


def saved_model(name=SAVED_MODEL):
    project = _project()
    return project.get_saved_model(next(m["id"] for m in project.list_saved_models() if m["name"] == name))


def forecast_model(forecaster):
    """An MLflow model around a trained forecaster. The nightly forecast unwraps it and calls the forecaster
    (forecast.production_forecaster); called as an MLflow model, it takes daily flows (interface 1) and returns the
    forecast of the 3 windows after the last day it was given, without client details."""
    import mlflow.pyfunc
    from models import forecast

    class ForecastModel(mlflow.pyfunc.PythonModel):
        def __init__(self, forecaster):
            self.forecaster = forecaster

        def predict(self, context, model_input, params=None):
            cutoff = pd.to_datetime(model_input["day"]).max().normalize()
            return forecast(self.forecaster, model_input, None, cutoff)

    return ForecastModel(forecaster)


def _carry_code(forecaster):
    """The models carry their own copy of the library code (pickled by value: cashflow, models, tracking and the
    forecaster's own module, forecasters.py for example), so they keep working when the library changes."""
    import cloudpickle
    import cashflow
    import models
    own = sys.modules.get(type(forecaster).__module__)
    for module in (cashflow, models, sys.modules[__name__], own):
        if module is not None and module.__name__ != "__main__":
            cloudpickle.register_pickle_by_value(module)


def _record(mlflow, params, metrics, backtest):
    """A run's settings, test scores and backtest forecaster."""
    import cloudpickle
    mlflow.log_params({k: str(v)[:250] for k, v in params.items()})
    mlflow.log_metrics({k: float(v) for k, v in metrics.items() if pd.notna(v)})
    with tempfile.TemporaryDirectory() as tmp:
        with open(os.path.join(tmp, "forecaster.pkl"), "wb") as f:
            cloudpickle.dump(backtest, f)
        mlflow.log_artifacts(tmp, artifact_path="backtest")


def log_model(model_id, params, metrics, backtest, final):
    """Record one training run in Experiment Tracking, then deploy its final forecaster as version model_id of the
    saved model cash_forecast (not activated: the promotion does that)."""
    _carry_code(final)
    with _mlflow(FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, EXPERIMENT, FOLDER))
        with mlflow.start_run(run_name=model_id) as run:
            mlflow.set_tag("model_id", model_id)
            _record(mlflow, params, metrics, backtest)
            where = "name" if int(mlflow.__version__.split(".")[0]) >= 3 else "artifact_path"   # MLflow 3 or 2
            mlflow.pyfunc.log_model(python_model=forecast_model(final), **{where: "forecaster"})
            run_id = run.info.run_id
    extension = _project().get_mlflow_extension()
    code_env = _project().get_recipe("train_and_test").get_settings().get_recipe_params()["envSelection"]["envName"]
    extension.set_run_inference_info(run_id=run_id, prediction_type="REGRESSION", code_env_name=code_env)
    # which model of the run: model_name in DSS 15 (MLflow 3 models), model_sub_folder in DSS 14 (the artifact path)
    which = "model_name" if "model_name" in inspect.signature(extension.deploy_run_model).parameters else "model_sub_folder"
    model = saved_model()
    before = (model.get_active_version() or {}).get("id")
    extension.deploy_run_model(run_id, model.id, version_id=model_id, activate=False, skip_expensive_reports=True,
                               **{which: "forecaster"})
    if before and (model.get_active_version() or {}).get("id") != before:
        model.set_active_version(before)       # DSS 14 activates every deployed version: only a promotion may


def log_lab_run(run_name, params, metrics, backtest):
    """Record a lab try in Experiment Tracking (experiment cashflow_lab, folder experiments): its settings, test
    scores and forecaster. Never deployed. Returns the run ID."""
    _carry_code(backtest)
    with _mlflow(LAB_FOLDER) as mlflow:
        mlflow.set_experiment(_experiment_name(mlflow, LAB_EXPERIMENT, LAB_FOLDER))
        with mlflow.start_run(run_name=run_name) as run:
            _record(mlflow, params, metrics, backtest)
            return run.info.run_id


def runs():
    """Every training run, newest first: model_id, run_id, trained_at, its settings (params.*) and metrics."""
    with _mlflow(FOLDER) as mlflow:
        # sorted here: DSS 14 cannot sort runs by start time in the search
        found = mlflow.search_runs(experiment_names=[_experiment_name(mlflow, EXPERIMENT, FOLDER)])
    if found.empty:
        return pd.DataFrame(columns=["model_id", "run_id", "trained_at"])
    found = found.sort_values("start_time", ascending=False).reset_index(drop=True)
    return found.rename(columns={"tags.model_id": "model_id", "start_time": "trained_at"})


def latest_model_id():
    """The most recently trained model."""
    found = runs()
    if found.empty:
        raise ValueError("No model trained yet: run the scenario Retrain, test and promote")
    return found.iloc[0]["model_id"]


def production_model_id():
    """The production model: the active version of the saved models, if it was ever promoted (a version Dataiku
    activated on its own when the first one was deployed does not count until the promotion)."""
    model_id = (saved_model().get_active_version() or {}).get("id")
    return model_id if model_id in promoted_model_ids() else ""


def promoted_model_ids():
    """Every model ID promoted so far (read through the API: also works in a recipe that writes promotion_log)."""
    try:
        log = _project().get_dataset("promotion_log")
        columns = [c["name"] for c in log.get_schema()["columns"]]
        model, decision = columns.index("candidate_model_id"), columns.index("decision")
        return {row[model] for row in log.iter_rows() if row[decision] == "PROMOTE"}
    except Exception:            # no promotion yet
        return set()


def load_backtest(model_id):
    """The backtest forecaster of a training run (trained without the last 6 periods, for champion / challenger)."""
    found = runs()
    match = found[found["model_id"] == model_id]
    if match.empty:
        raise ValueError(f"Model {model_id} not found in Experiment Tracking ({EXPERIMENT})")
    with _mlflow(FOLDER) as mlflow:
        path = mlflow.artifacts.download_artifacts(run_id=match.iloc[0]["run_id"], artifact_path="backtest")
    with open(os.path.join(path, "forecaster.pkl"), "rb") as f:
        return pickle.load(f)


def decide(candidate, production, rules=None):
    """Champion / challenger: the decision (PROMOTE or KEEP) and its reason, from the test scores of the candidate and
    of the production model (models.scores), with the promotion rules (promotion_rules: project variables). production
    None: no production model to compare with, the candidate is promoted (the forecast needs one) with its failed
    checks as a warning."""
    rules = rules or promotion_rules()
    low, high = rules["promotion_min_coverage"], rules["promotion_max_coverage"]
    checks = []
    if not low <= candidate["coverage_p10_p90"] <= high:
        checks.append(f"coverage {candidate['coverage_p10_p90']:.1%} outside {low:.0%}-{high:.0%}")
    if rules["promotion_beat_simple_guess"] and candidate["wape_model"] >= candidate["wape_baseline"]:
        checks.append(f"does not beat the simple guess ({candidate['wape_model']:.1%} vs "
                      f"{candidate['wape_baseline']:.1%})")
    if production is None:
        return "PROMOTE", (f"no production model to compare with, EV error {candidate['wape_model']:.2%}, coverage "
                           f"{candidate['coverage_p10_p90']:.1%}" + (f" (warning: {'; '.join(checks)})" if checks else ""))
    if checks:
        return "KEEP", f"candidate {checks[0]}"
    needed = production["wape_model"] * (1 - rules["promotion_min_gain"])
    if candidate["wape_model"] >= needed:
        gain = f" by more than {rules['promotion_min_gain']:.0%}" if rules["promotion_min_gain"] else ""
        return "KEEP", (f"candidate EV error {candidate['wape_model']:.2%} not lower than production "
                        f"{production['wape_model']:.2%}{gain}")
    return "PROMOTE", (f"candidate EV error {candidate['wape_model']:.2%} vs production {production['wape_model']:.2%}, "
                       f"coverage {candidate['coverage_p10_p90']:.1%}")


def apply_promotion():
    """Apply the latest decision in promotion_log: on PROMOTE, activate the candidate's version of the saved model.
    Run by a scenario step after the champion / challenger test."""
    import dataiku
    log = dataiku.Dataset("promotion_log").get_dataframe()
    last = log.sort_values("decided_at").iloc[-1]
    if last["decision"] == "PROMOTE":
        saved_model().set_active_version(last["candidate_model_id"])
        print(f"Promoted {last['candidate_model_id']}: {last['reason']}")
    else:
        print(f"Kept the production model: {last['reason']}")
    return production_model_id()
'''

In [ ]:
FORECAST_CODE = r'''
"""Inference (project library: python/forecast.py): the production model and the final forecast table. Inference reads
nothing from training except the active version of the saved model cash_forecast (interface 3, cashflow.SAVED_MODEL).
"""
import numpy as np
import pandas as pd

from cashflow import HORIZONS, MIN_HISTORY_DAYS, SAVED_MODEL, TOTAL, window

VALUES = ("p10", "ev", "p90")


def _saved_model_id():
    import dataiku
    project = dataiku.api_client().get_default_project()
    ids = {m["name"]: m["id"] for m in project.list_saved_models()}
    if SAVED_MODEL not in ids:
        raise ValueError(f"Saved model {SAVED_MODEL} not found. Run the setup notebook.")
    return ids[SAVED_MODEL]


def active_model_id():
    """The production model: the active version of the saved model cash_forecast."""
    import dataiku
    model = dataiku.api_client().get_default_project().get_saved_model(_saved_model_id())
    model_id = (model.get_active_version() or {}).get("id")
    if not model_id:
        raise ValueError("No production model yet: run the scenario Retrain, test and promote")
    return model_id


def production_forecaster():
    """The production model's trained forecaster (the active version of cash_forecast), and its model ID. The recipe
    that calls it must have the saved model as an input."""
    import dataiku
    model_id = active_model_id()
    predictor = dataiku.Model(_saved_model_id()).get_predictor(model_id)
    return predictor.model.unwrap_python_model().forecaster, model_id


def forecast_table(pred, history, cutoff):
    """The forecast: one row per client, category (and TOTAL) and window (1, 2, 3: days 1-30, 31-60, 61-90 after the
    cut-off day), the net cash flow (money in minus money out) as P10, EV and P90 in SAR. pred: the production model's
    forecast (models.forecast); history: days of history per client (cashflow.history_days). A client with less than
    MIN_HISTORY_DAYS days of history gets no values (status NOT_ENOUGH_HISTORY)."""
    frame = pred.merge(history.reset_index(), on="client_no", how="left")
    enough = frame["history_days"].fillna(0) >= MIN_HISTORY_DAYS
    frame.loc[~enough, ["p10", "ev", "p90"]] = np.nan
    frame["status"] = np.where(enough, "OK", "NOT_ENOUGH_HISTORY")
    return frame.sort_values(["client_no", "category", "window"]).reset_index(drop=True)


def value_columns():
    """The value columns of the final forecast, always the same and in this order: TOTAL first, then each category of
    warehouse.py (CATEGORY_OF_TYPE and OTHER), each for windows 1, 2, 3 (days 1-30, 31-60, 61-90 after the cut-off
    day) and P10, EV, P90: TOTAL_w1_p10, TOTAL_w1_ev, TOTAL_w1_p90, TOTAL_w2_p10, ..., UTILITIES_w3_p90."""
    from warehouse import CATEGORY_OF_TYPE, OTHER
    categories = [TOTAL] + sorted(set(CATEGORY_OF_TYPE.values()) | {OTHER})
    return [f"{c}_w{h}_{v}" for c in categories for h in HORIZONS for v in VALUES]


def one_row_per_client(table, cutoff, no_postings=()):
    """The final forecast: ONE row per client. table: forecast_table (one row per client, category and window).
    Columns: client_no, cutoff_date (the last day of data), w1_start, w1_end, w2_start, w2_end, w3_start, w3_end (the
    days of each window), status, history_days, then value_columns() (empty for a category the client does not have,
    and for a client without enough history). no_postings: active clients with no posting at all, added with status
    NO_POSTINGS and no values."""
    columns = value_columns()
    keys = ["client_no", "category", "window"]
    values = pd.DataFrame(index=pd.Index([], name="client_no"))
    if len(table):
        values = table.set_index(keys)[list(VALUES)].unstack(["category", "window"])
        values.columns = [f"{c}_w{h}_{v}" for v, c, h in values.columns]
    unknown = sorted(set(values.columns) - set(columns))
    if unknown:
        raise ValueError(f"Forecast for categories not in warehouse.py: {', '.join(sorted({u.split('_w')[0] for u in unknown}))}")
    info = table.groupby("client_no")[["status", "history_days"]].first()
    missing = pd.DataFrame({"status": "NO_POSTINGS", "history_days": 0},
                           index=pd.Index(sorted(no_postings), name="client_no"))
    info = pd.concat([info, missing])
    info["history_days"] = pd.to_numeric(info["history_days"]).fillna(0).astype(int)
    days = {"cutoff_date": cutoff}
    for h in HORIZONS:
        days[f"w{h}_start"], days[f"w{h}_end"] = window(cutoff, h)
    return pd.concat([pd.DataFrame(days, index=info.index), info,
                      values.reindex(index=info.index, columns=columns).astype(float)], axis=1).reset_index()
'''

In [ ]:
FORECASTERS_CODE = r'''
"""The data scientist's forecasters (project library: python/forecasters.py). The setup notebook creates this file once
and never overwrites it: it is yours.

Try a forecaster first in the notebook forecast_lab (it never touches production). To put one in production:
    1. add its class here and to FORECASTERS below,
    2. set the project variable forecaster to its name,
    3. run the scenario Retrain, test and promote: champion / challenger promotes it only if it passes the promotion
       rules (project variables promotion_*) on the same last 6 periods of 30 days. To go back, set forecaster to quantile_boosting.

A forecaster is a class with (details in models.py, and the examples QuantileBoosting and RecentAverage there):
    name            the value of the project variable forecaster that selects it
    params          its settings (a dict), recorded in Experiment Tracking
    fit(daily, clients, cutoff)
                    learns from the daily flows (client_no, day, category, money_in, money_out, postings) up to the
                    cut-off day (yesterday), and the clients
    predict(daily, clients, cutoff)
                    one row per client, category (and TOTAL) and window (1, 2, 3: days 1-30, 31-60, 61-90 after
                    the cut-off day): client_no, category, window, p10, ev, p90 (the net cash flow of that window, SAR)
cashflow.period_matrix(daily, cutoff) gives the net cash flow per client and category (and TOTAL) and 30-day period
ending at the cut-off day.
"""
from models import RecentAverage


class RecentAverage6(RecentAverage):
    """Example: recent_average (models.py) with the average of the last 6 periods of 30 days instead of 3."""
    name = "recent_average_6m"
    params = {**RecentAverage.params, "periods": 6}


FORECASTERS = [RecentAverage6]
'''

In [ ]:
RECIPES = {}

In [ ]:
RECIPES["stage_tables"] = r'''
# Staging: the 2 warehouse tables, read-only, turned into the project's tables, all in Dataiku. This is interface 1,
# what the data scientist's models get:
#   clients: the business clients of DIM_CUSTOMER (companies and sole proprietors), standard names and values. Read
#   whole every run (it is small).
#   daily_flows: the postings of FCT_STATEMENT_ENTRY summed per client, day and category (money in, money out):
#   business clients only, no reversals, no transfers between the client's own accounts. Only the NEW days are read
#   from the warehouse: the days after the last day already staged, up to the cut-off day (yesterday). When nothing
#   is staged yet (the first run, or after the scenario Full rebuild), the whole history is read.
#   daily_flows is a folder of Parquet files (read straight from the files, far faster than a dataset), kept GROUPED
#   BY CLIENT: each file holds complete clients (sorted by client, day, category). So when there are new days, it is
#   written again as a new version: the rows already staged and the new ones, a group of clients per file; the new
#   version only becomes the current one once every file is written (cashflow.write_daily), so a failed run leaves
#   daily_flows as it was. Inference (and anyone else) reads it in chunks of complete clients. When there is nothing
#   new, daily_flows is not touched.
#   staging_log: one row per run (appended), also when the run fails (status FAILED and the error): what was read,
#   and the last day now in daily_flows.
# The warehouse does the heavy work (warehouse.daily_totals_query): it sums the postings into daily totals per client,
# day and category, without reversals and own-account transfers, so only those totals travel, not every posting. A
# big read never sits in memory whole: the totals read and the rows already staged are spread over files on disk by
# client and written one file (a group of clients) at a time.
import json
import os
import tempfile
import time

import dataiku
import pandas as pd

from cashflow import (DAILY_COLUMNS, Progress, Spread, append_row, clients_limit, cutoff_date, history_start,
                      iter_daily_parts, waiting, write_daily)
import warehouse
from warehouse import TABLES, daily_totals_query, one_row_per_client, standard_daily, table_query, to_standard

CHUNK = 200_000
STEP = 10_000                       # rows: how often the reading progress is counted (CHUNK is a multiple of it)
ROWS_PER_GROUP = 1_000_000          # about this many daily_flows rows per group of clients when rewriting
GROUPS_FULL = 512                   # groups of clients when everything is read (its size is not known before)
KEYS = ["client_no", "day", "category"]
started = time.time()
cutoff = cutoff_date()
end = cutoff + pd.Timedelta(days=1)                # postings up to the cut-off day (included)


def read(query, progress, convert):
    """Runs a read-only query through the warehouse connection and yields its rows in chunks (DataFrames, converted
    by convert). progress counts the rows as they arrive. The query is printed first, and until its first row
    arrives a line says every minute that the warehouse is still running it."""
    print(f"PROGRESS SQL: {query}", flush=True)
    started, first = time.time(), waiting(progress.what)
    try:
        reader = dataiku.SQLExecutor2(dataset=dataiku.Dataset(TABLES["clients"][0])).query_to_iter(query)
        names, rows = [c["name"] for c in reader.get_schema()], []
        for row in reader.iter_tuples():
            if not first.is_set():
                first.set()
                print(f"PROGRESS {progress.what}: first rows after {(time.time() - started) / 60:.1f} min",
                      flush=True)
            rows.append(row)
            if len(rows) % STEP == 0:
                progress.add(STEP)
            if len(rows) == CHUNK:
                yield convert(pd.DataFrame(rows, columns=names))
                rows = []
        progress.add(len(rows) % STEP)
        if rows:
            yield convert(pd.DataFrame(rows, columns=names))
    finally:
        first.set()


def table_name(table):
    return sql_name(dataiku.Dataset(TABLES[table][0]))


def sql_name(dataset):
    """The table's full quoted name. The project reads the warehouse through SQL only: anything else stops here."""
    try:
        name = dataset.get_location_info()["info"].get("quotedResolvedTableName")
    except Exception:
        name = None
    if not name:
        raise ValueError(f"{dataset.name} is not a database table: the project reads the warehouse through SQL only. "
                         "Run the setup notebook again (it links the warehouse tables)")
    return name


LOG = {"run_at": "string", "status": "string", "error": "string", "mode": "string", "cutoff_date": "string",
       "read_from": "string", "read_to": "string", "postings_read": "bigint", "postings_kept": "bigint",
       "daily_rows_added": "bigint", "daily_rows_total": "bigint", "last_day": "string", "postings_per_month": "string",
       "clients": "bigint", "seconds": "double"}
row = {"run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"), "cutoff_date": f"{cutoff:%Y-%m-%d}",
       "read_to": f"{cutoff:%Y-%m-%d}"}


def stage():
    """Reads the new days and adds them to daily_flows; fills row (the staging_log row) as it goes."""
    # what is already staged: staging_log (this recipe adds to it, so it reads it outside the flow's inputs). Failed
    # runs changed nothing.
    try:
        log = dataiku.Dataset("staging_log", ignore_flow=True).get_dataframe()
    except Exception:
        log = pd.DataFrame(columns=list(LOG))
    log = log[log["status"] == "SUCCESS"] if len(log) else log
    last_day = pd.to_datetime(log["last_day"]).max() if len(log) else None
    staged_rows = int(log.sort_values("run_at")["daily_rows_total"].iloc[-1]) if len(log) else 0
    if last_day is None:
        if next(iter_daily_parts(ignore_flow=True), None) is not None:
            raise ValueError("staging_log is empty but daily_flows is not: run the scenario Full rebuild")
    since = None if last_day is None else last_day + pd.Timedelta(days=1)
    mode = "FULL" if since is None else "NOTHING_NEW" if since >= end else "NEW_DAYS"
    if mode == "FULL":
        since = history_start(cutoff)              # project variable history_months (None: the whole history)
    row.update(mode=mode, read_from=f"{since:%Y-%m-%d}" if since is not None and mode != "NOTHING_NEW" else "")

    limit = clients_limit()                        # project variable clients_limit (None: all the business clients)
    print(f"PROGRESS staging ({mode})" + (f", the first {limit:,} business clients only" if limit else ""), flush=True)
    progress = Progress("reading the clients from the warehouse (rows)")
    clients = one_row_per_client(pd.concat(
        [one_row_per_client(chunk) for chunk in read(table_query("clients", table_name("clients"), limit=limit), progress,
                                                   lambda df: to_standard(df, "clients"))], ignore_index=True))
    progress.finish()
    print(f"PROGRESS {len(clients):,} business clients", flush=True)
    if progress.done > len(clients):               # DIM_CUSTOMER keeps history: several rows per client
        print(f"WARNING: DIM_CUSTOMER sent {progress.done:,} rows for {len(clients):,} business clients (several "
              "rows per client). " + ("Its condition CLIENTS_CURRENT_ROW does not keep one row per client: check it"
                                      if warehouse.CLIENTS_CURRENT_ROW.strip() else
                                      "Fill in CLIENTS_CURRENT_ROW in warehouse.py (the notebook's WAREHOUSE_CODE "
                                      "cell) to read only each client's current row")
              + ". Until then, the last row received is kept, which may not be the newest.", flush=True)
    if clients.empty:
        raise ValueError("No business client in DIM_CUSTOMER: check CLIENTS_BUSINESS and CLIENTS_CURRENT_ROW in "
                         "warehouse.py (the notebook's WAREHOUSE_CODE cell)")
    numbers = set(clients["client_no"])
    dataiku.Dataset("clients").write_with_schema(clients)
    row["clients"] = len(clients)

    read_rows = kept = added = total = 0          # postings in the daily totals read, and kept
    new = {}                                       # postings per month read now
    with tempfile.TemporaryDirectory() as spread:
        if mode != "NOTHING_NEW":
            groups = Spread(os.path.join(spread, "groups"),     # groups of clients
                            GROUPS_FULL if mode == "FULL" else max(16, -(-staged_rows // ROWS_PER_GROUP) + 1))
            query = daily_totals_query(table_name("transactions"), table_name("clients"), since, end, limit)
            progress = Progress("reading the daily totals from the warehouse (rows: client x day x category)")
            for chunk in read(query, progress, standard_daily):
                read_rows += int(chunk["postings"].sum())
                chunk = chunk[chunk["client_no"].isin(numbers)]
                kept += int(chunk["postings"].sum())
                if len(chunk):
                    last_day = max(chunk["day"].max(), last_day or pd.Timestamp.min)
                    for month, n in chunk.groupby(chunk["day"].dt.strftime("%Y-%m"))["postings"].sum().items():
                        new[month] = new.get(month, 0) + int(n)
                    groups.add(chunk[DAILY_COLUMNS], "client_no")
            progress.finish()
            row.update(postings_read=read_rows, postings_kept=kept)

        # the postings must reach the cut-off day (yesterday): otherwise the warehouse load looks late. Checked before
        # anything is written, so a failed run changes nothing and the next run reads the same days again.
        if last_day is None or last_day < cutoff:
            raise ValueError(f"FCT_STATEMENT_ENTRY has no posting of business clients after "
                             f"{'(none)' if last_day is None else f'{last_day:%Y-%m-%d}'}: the warehouse load of "
                             f"{cutoff:%Y-%m-%d} (the cut-off day) looks late")

        if mode == "NOTHING_NEW":
            total = staged_rows                    # daily_flows is not touched
        else:
            # the rows already staged join the new ones, by client; then daily_flows is written again as a new
            # version, a group of clients per file, each client's rows together
            old = 0
            if mode == "NEW_DAYS":
                progress = Progress("adding the rows already in daily_flows", staged_rows)
                for part in iter_daily_parts(ignore_flow=True):
                    groups.add(part[DAILY_COLUMNS], "client_no")
                    old += len(part)
                    progress.add(len(part))
                progress.finish()
            if old != staged_rows:
                raise ValueError(f"daily_flows has {old:,} rows but staging_log says {staged_rows:,}: run the "
                                 "scenario Full rebuild of the daily flows")
            progress = Progress("writing daily_flows (groups of clients)", groups.n)
            total = write_daily(counted(part.groupby(KEYS, as_index=False)[["money_in", "money_out", "postings"]]
                                        .sum().round({"money_in": 2, "money_out": 2}).sort_values(KEYS), progress)
                                for part in groups.parts())
            progress.finish()
            added = total - old
    row.update(postings_read=read_rows, postings_kept=kept, daily_rows_added=added, daily_rows_total=total,
               last_day=f"{last_day:%Y-%m-%d}", postings_per_month=json.dumps(dict(sorted(new.items()))))


def counted(part, progress):
    progress.add()
    return part


def write_log(status, error=""):
    row.update(status=status, error=error[:1000], seconds=round(time.time() - started, 1))
    append_row("staging_log", LOG, row)


try:
    stage()
except Exception as e:
    write_log("FAILED", f"{type(e).__name__}: {e}")
    raise
write_log("SUCCESS")
print(f"{row['mode']}: {row['clients']:,} business clients; {row['postings_read']:,} postings read"
      + (f" from {row['read_from']}" if row["read_from"] else "") + f", {row['postings_kept']:,} kept; "
      f"{row['daily_rows_added']:,} daily totals added, {row['daily_rows_total']:,} in daily_flows; last day staged "
      f"{row['last_day']}")
'''

In [ ]:
RECIPES["train_and_test"] = r'''
# Train and test one model, with its own model ID, tracked in Experiment Tracking with its forecaster, settings and
# scores. The forecaster is the one the project variable forecaster names (default quantile_boosting, in models.py; the
# data scientist's own are in forecasters.py). It learns from the daily flows and the clients (interface 1).
#   backtest: trained without the last 6 periods of 30 days, then tested on them (models.backtest, test_results)
#   final: trained on everything up to the cut-off, deployed as version <model ID> of the saved model cash_forecast
#   (not active: the promotion activates it)
import time

import dataiku
import pandas as pd

from cashflow import MIN_TRAINING_MONTHS, cutoff_date, read_daily, read_dataset, until
from models import backtest, new_forecaster, run_metrics, test_results
from tracking import log_model, new_model_id

started = time.time()


def step(what):
    """A progress line in the job's log (the setup notebook shows it while it waits)."""
    print(f"PROGRESS training: {what} ({(time.time() - started) / 60:.1f} min so far)", flush=True)


cutoff = cutoff_date()
step("reading daily_flows and clients")
daily, clients = until(read_daily(), cutoff), read_dataset("clients")
first_day = pd.to_datetime(daily["day"]).min() if len(daily) else cutoff
if first_day > cutoff - pd.DateOffset(months=MIN_TRAINING_MONTHS) + pd.Timedelta(days=1):
    raise ValueError(f"Not enough history to train and test a model: daily_flows starts on {first_day:%Y-%m-%d}, and "
                     f"a model needs at least {MIN_TRAINING_MONTHS} months up to the cut-off day ({cutoff:%Y-%m-%d}). "
                     "Run the setup again with more HISTORY_MONTHS, then the scenario Full rebuild")
backtested, final = new_forecaster(), new_forecaster()
step(f"1/2 test: training without the last periods, then testing on them ({len(daily):,} daily rows)")
tested = backtest(backtested, daily, clients, cutoff)
step("2/2 training the final model on everything up to the cut-off")
final.fit(daily, clients, cutoff)
step("saving and deploying the model")
model_id = new_model_id(cutoff)

results = test_results(tested)
results.insert(0, "model_id", model_id)
metrics = run_metrics(results)
params = {"forecaster": final.name, **getattr(final, "params", {}), "cutoff_date": f"{cutoff:%Y-%m-%d}",
          "load_id": dataiku.get_custom_variables().get("load_id", ""), "daily_rows": len(daily),
          "clients": daily["client_no"].nunique(), "test_rows": len(tested)}
log_model(model_id, params, metrics, backtested, final)
print(f"Model {model_id} ({final.name}): EV error {metrics['wape']:.1%} (simple guess {metrics['wape_baseline']:.1%}), "
      f"coverage {metrics['coverage_p10_p90']:.1%}")

dataiku.Dataset("test_results").write_with_schema(results)
'''

In [ ]:
RECIPES["champion_challenger"] = r'''
# Champion / challenger: the latest trained model (candidate) against the production model, both tested on the
# same last 6 periods of 30 days with their backtest forecasters (trained without those periods). The rules are project variables
# (tracking.PROMOTION_RULES, applied by tracking.decide): promoted if its P10-P90 coverage is in the range
# (default 70%-90%), it beats the simple guess (the same as the month before) and its EV error is lower than the
# production model's (by more than promotion_min_gain; a tie keeps the production model). If there is no production
# model to compare with (none yet, or its tracked files were deleted), the candidate is promoted (the forecast needs
# one), with its failed checks as a warning. The production model is the active version of the saved model
# cash_forecast. The decision is added to promotion_log; the next scenario step applies it (activates the
# candidate's version).
import dataiku
import pandas as pd

from cashflow import cutoff_date, read_daily, read_dataset, until
from models import scores, test_forecasts
from tracking import decide, latest_model_id, load_backtest, production_model_id, promotion_rules, runs

cutoff = cutoff_date()
daily, clients = until(read_daily(), cutoff), read_dataset("clients")
candidate = latest_model_id()
production = production_model_id()
tracked = runs()
if production and production not in set(tracked["model_id"]):     # its tracked files were deleted: nothing to test
    print(f"Production model {production} is not in Experiment Tracking: the candidate is judged on its own")
    production = ""
tested = {m: scores(test_forecasts(load_backtest(m), daily, clients, cutoff)) for m in {candidate, production} if m}
c = tested[candidate]
p = tested.get(production) if production and production != candidate else None
if production == candidate:
    decision, reason = "KEEP", "the candidate is already the production model"
else:
    decision, reason = decide(c, p, promotion_rules())
print(decision, "-", reason)

forecaster = dict(zip(tracked["model_id"], tracked["params.forecaster"])) if "params.forecaster" in tracked else {}
dataiku.Dataset("promotion_log").write_with_schema(pd.DataFrame([{
    "decided_at": pd.Timestamp.now().floor("s"), "cutoff_date": dataiku.get_custom_variables()["cutoff_date"],
    "candidate_model_id": candidate, "candidate_forecaster": forecaster.get(candidate),
    "production_model_id": production or None, "production_forecaster": forecaster.get(production),
    "candidate_wape": c["wape_model"], "production_wape": p["wape_model"] if p is not None else None,
    "baseline_wape": c["wape_baseline"], "candidate_coverage": c["coverage_p10_p90"],
    "production_coverage": p["coverage_p10_p90"] if p is not None else None,
    "decision": decision, "reason": reason, "rules": str(promotion_rules())}]))
'''

In [ ]:
RECIPES["final_forecast"] = r'''
# Final forecast: the production model (the active version of the saved model cash_forecast) forecasts, for each
# active business client, the net cash flow (money in minus money out) of each of the next 3 windows of 30 days after
# the cut-off day (yesterday): days 1-30, 31-60 and 61-90, per category and in TOTAL, as P10, EV and P90 in SAR, from the client's daily flows (interface 1). Inference reads nothing else from
# training. final_forecast has ONE row per active client (forecast.one_row_per_client): status OK, NOT_ENOUGH_HISTORY
# (no values) or NO_POSTINGS (no posting at all, no values), the values as columns (TOTAL_w1_p10, TOTAL_w1_ev, ...),
# and which model (model_id, forecaster) and which data (load_id) produced it.
# Built to scale: staging keeps daily_flows grouped by client, so it is read straight through in chunks of complete
# clients (cashflow.iter_clients: about inference_clients_per_chunk clients each, project variable) and each chunk is
# forecast as soon as it is read. Memory holds one chunk of clients, whatever their number. The chunks' forecasts are
# kept on local disk, then, once every chunk has worked, written to:
#   final_forecast: the forecast
#   forecast_archive: one compressed CSV file per cut-off day and model (forecast_<cut-off>_<model ID>.csv.gz)
#   inference_log: one row per run (appended), also when the run fails (status FAILED and the error): which model and
#   load, how many active clients got a forecast or not and why, how long each step took, the most memory used
import gzip
import os
import resource
import tempfile
import time

import dataiku
import pandas as pd

from cashflow import Progress, append_row, cutoff_date, history_days, iter_clients, read_dataset, until
from forecast import forecast_table, one_row_per_client, production_forecaster
from models import active, forecast

LOG = {"run_at": "string", "status": "string", "error": "string", "load_id": "string", "as_of_date": "string",
       "cutoff_date": "string", "model_id": "string", "forecaster": "string", "active_clients": "bigint",
       "clients_ok": "bigint", "clients_not_enough_history": "bigint", "clients_no_postings": "bigint",
       "daily_rows_read": "bigint", "chunks": "bigint", "seconds_reading": "double", "seconds_forecasting": "double",
       "seconds_writing": "double", "seconds_total": "double", "peak_memory_mb": "bigint", "archive_file": "string"}
NO_ROWS = ["client_no", "category", "window", "p10", "ev", "p90", "status", "history_days"]
started = time.time()
variables = dataiku.get_custom_variables()
log = {"run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"), "load_id": variables.get("load_id", ""),
       "as_of_date": variables.get("as_of_date", ""), "cutoff_date": variables.get("cutoff_date", "")}


def run():
    """The forecast; fills log as it goes, so a failed run still logs what it got to."""
    cutoff = cutoff_date()
    forecaster, model_id = production_forecaster()
    log.update(model_id=model_id, forecaster=getattr(forecaster, "name", type(forecaster).__name__))
    _, clients = active(pd.DataFrame(columns=["client_no"]), read_dataset("clients"))     # the active clients
    numbers = set(clients["client_no"])
    if not numbers:
        raise ValueError("No active client in clients: check the staging and STS_FLG in warehouse.py")
    per_chunk = int(variables.get("inference_clients_per_chunk") or 5000)
    name = f"forecast_{cutoff:%Y-%m-%d}_{model_id}.csv.gz"
    log.update(active_clients=len(numbers), archive_file=name)

    with tempfile.TemporaryDirectory() as folder:
        # read and forecast chunk by chunk; the forecasts are kept on disk: final_forecast is only written once every
        # chunk worked, so a failed run leaves the previous forecast as it was
        done, seen, daily_rows, forecasting = [], set(), 0, 0.0
        progress = Progress(f"forecasting the active clients ({per_chunk:,} per chunk)", len(numbers))
        for daily in iter_clients("daily_flows", per_chunk,
                                  keep=lambda d: until(d[d["client_no"].isin(numbers)], cutoff)):
            ticked = time.time()
            group = clients[clients["client_no"].isin(set(daily["client_no"]))]
            pred = forecast(forecaster, daily, group, cutoff)
            table = forecast_table(pred, history_days(daily, cutoff), cutoff)
            done.append(keep(one_row_per_client(table, cutoff), folder, len(done)))
            seen |= set(group["client_no"])
            daily_rows += len(daily)
            forecasting += time.time() - ticked
            progress.add(len(group))
        missing = numbers - seen                     # active clients with no posting at all: one row each
        progress.add(len(missing))
        progress.finish()
        if missing:
            done.append(keep(one_row_per_client(pd.DataFrame(columns=NO_ROWS), cutoff, missing), folder,
                             len(done)))
        read_done = time.time()
        log.update(daily_rows_read=daily_rows, chunks=len(done) - bool(missing), seconds_forecasting=round(forecasting, 1),
                   seconds_reading=round(read_done - started - forecasting, 1))

        out, writer, counts = dataiku.Dataset("final_forecast"), None, {}
        progress = Progress("writing final_forecast and the archive (chunks)", len(done))
        with dataiku.Folder("forecast_archive").get_writer(name) as archive_file, \
                gzip.GzipFile(fileobj=archive_file, mode="wb") as archive:
            for path in done:
                frame = pd.read_pickle(path)
                frame["model_id"], frame["forecaster"] = model_id, log["forecaster"]
                frame["load_id"] = variables.get("load_id", "")
                frame = frame.round(2)
                if writer is None:
                    out.write_schema_from_dataframe(frame)
                    writer = out.get_writer()
                writer.write_dataframe(frame)
                archive.write(frame.to_csv(index=False, header=path == done[0]).encode())
                for status, n in frame["status"].value_counts().items():
                    counts[status] = counts.get(status, 0) + int(n)
                progress.add()
        writer.close()
        progress.finish()
    log.update(clients_ok=counts.get("OK", 0), clients_not_enough_history=counts.get("NOT_ENOUGH_HISTORY", 0),
               clients_no_postings=counts.get("NO_POSTINGS", 0), seconds_writing=round(time.time() - read_done, 1))


def keep(frame, folder, n):
    """A chunk's forecast, kept on local disk until every chunk is done. Returns its path."""
    path = os.path.join(folder, f"forecast_{n}.pkl")
    frame.to_pickle(path)
    return path


def write_log(status, error=""):
    log.update(status=status, error=error[:1000], seconds_total=round(time.time() - started, 1),
               peak_memory_mb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024))
    append_row("inference_log", LOG, log)


try:
    run()
except Exception as e:
    write_log("FAILED", f"{type(e).__name__}: {e}")
    raise
write_log("SUCCESS")
print(f"Forecast with {log['model_id']} ({log['forecaster']}): {log['active_clients']:,} active clients in "
      f"{log['chunks']} chunks: {log['clients_ok']:,} OK, {log['clients_not_enough_history']:,} without enough "
      f"history, {log['clients_no_postings']:,} without postings; {log['seconds_total']:.0f} s; "
      f"archived {log['archive_file']}")
'''

In [ ]:
RECIPES["load_test_data"] = r'''
# Load test, step 1 of 2: fake data, the same shape as the real data (interface 1), in the Load test zone only.
# It never reads or writes the real data: it writes
#   load_test_clients: one row per fake client (the columns of clients, all ACTIVE)
#   load_test_flows: fake daily flows, Parquet files exactly like daily_flows (one row per client, day and category:
#                    money_in, money_out, postings; each client's rows in one file), and benchmark.json (what was
#                    generated and how long it took, for load_test_run's log)
# Its settings are variables, set in the first step of the scenario Load test (Scenarios > Load test > Steps):
#   loadtest_clients             how many fake clients
#   loadtest_months              months of history up to the cut-off day (yesterday)
#   loadtest_postings_per_month  the average number of postings (transactions) per client and month
# Each client gets its own rate around that average (a few busy clients, many quiet ones), and 10% of the clients a
# shorter history, so some get NOT_ENOUGH_HISTORY, as in production.
import json
import time

import dataiku
import numpy as np
import pandas as pd

from cashflow import Progress, whole_number, write_daily
from warehouse import CATEGORY_OF_TYPE, COLUMNS, OTHER

started = time.time()
variables = dataiku.get_custom_variables()


def setting(name, default):
    value = whole_number(name, variables.get(name, default))
    if value is None:
        raise ValueError(f"The variable {name} must be a number (1 or more): set it in the scenario Load test, "
                         "step 1")
    return value


CLIENTS = setting("loadtest_clients", 1000)
MONTHS = setting("loadtest_months", 24)
POSTINGS = setting("loadtest_postings_per_month", 40)
PER_FILE = 2000                                   # fake clients per Parquet file
cutoff = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
days = MONTHS * 30
categories = np.array(sorted(set(CATEGORY_OF_TYPE.values()) | {OTHER}))
rng = np.random.default_rng(7)
print(f"PROGRESS load test 1/2, fake data: {CLIENTS:,} clients x {MONTHS} months x about {POSTINGS} postings a "
      f"month = about {CLIENTS * MONTHS * POSTINGS:,} postings, up to {cutoff:%Y-%m-%d}", flush=True)

numbers = np.array([f"LT{i:08d}" for i in range(CLIENTS)])
clients = pd.DataFrame({column: "LOAD TEST" for column in COLUMNS["clients"]}, index=range(CLIENTS))
clients["client_no"], clients["status"], clients["sole_proprietor"] = numbers, "ACTIVE", False
dataiku.Dataset("load_test_clients").write_with_schema(clients)

totals = {"postings": 0, "rows": 0}


def files():
    """The fake daily flows, PER_FILE clients at a time (each client's rows together, as staging writes them)."""
    progress = Progress("load test 1/2, writing the fake daily flows (clients)", CLIENTS)
    for start in range(0, CLIENTS, PER_FILE):
        who = numbers[start:start + PER_FILE]
        n = len(who)
        rate = POSTINGS * rng.gamma(2.0, 0.5, n)                       # postings a month, average POSTINGS
        span = np.where(rng.random(n) < 0.1, rng.integers(30, days + 1, n), days)   # days of history
        count = rng.poisson(rate * span / 30)
        client = np.repeat(np.arange(n), count)
        size = rng.lognormal(8, 1.2, n)[client]                        # the client's typical amount (SAR)
        amount = size * rng.lognormal(0, 0.8, len(client))
        money_in = rng.random(len(client)) < 0.55
        postings = pd.DataFrame({
            "client": client,
            "back": (rng.random(len(client)) * span[client]).astype(int),     # days before the cut-off day
            "category": rng.integers(0, len(categories), len(client)),
            "money_in": np.where(money_in, amount, 0.0), "money_out": np.where(money_in, 0.0, amount)})
        daily = (postings.groupby(["client", "back", "category"], sort=True)
                 .agg(money_in=("money_in", "sum"), money_out=("money_out", "sum"), postings=("money_in", "size"))
                 .reset_index())
        daily = daily.sort_values(["client", "back"], ascending=[True, False])
        part = pd.DataFrame({"client_no": who[daily["client"].to_numpy()],
                             "day": (cutoff - pd.to_timedelta(daily["back"].to_numpy(), unit="D")).values,
                             "category": categories[daily["category"].to_numpy()],
                             "money_in": daily["money_in"].round(2).to_numpy(),
                             "money_out": daily["money_out"].round(2).to_numpy(),
                             "postings": daily["postings"].astype("int64").to_numpy()})
        totals["postings"] += len(postings)
        totals["rows"] += len(part)
        progress.add(n)
        yield part
    progress.finish()


write_daily(files(), "load_test_flows")
folder = dataiku.Folder("load_test_flows")
paths = [p for p in folder.list_paths_in_partition() if p.endswith(".parquet")]
size = sum(folder.get_path_details(p).get("size", 0) for p in paths)
info = {"cutoff": f"{cutoff:%Y-%m-%d}", "clients": CLIENTS, "months": MONTHS, "postings_per_month": POSTINGS,
        "postings": totals["postings"], "daily_rows": totals["rows"], "parquet_files": len(paths),
        "parquet_mb": round(size / 1e6, 1), "seconds_generating": round(time.time() - started, 1)}
with folder.get_writer("benchmark.json") as writer:
    writer.write(json.dumps(info).encode())
print(f"PROGRESS load test 1/2, fake data: done in {info['seconds_generating']:.0f} s: {info['postings']:,} postings, "
      f"{info['daily_rows']:,} daily rows, {info['parquet_files']} Parquet files ({info['parquet_mb']} MB)", flush=True)
'''

In [ ]:
RECIPES["load_test_run"] = r'''
# Load test, step 2 of 2: times the nightly forecast (like final_forecast) on the fake data of load_test_data, with
# a dummy model by default. It never reads or writes the real data, the real models or their logs. No real training:
# the model is fitted on 2 clients only (a few seconds), since only the forecast's time is measured:
#   the daily flows read in chunks of complete clients, each chunk forecast (one row per client), and each chunk
#   written to load_test_forecast (the columns of final_forecast)
# load_test_log: one row per run (appended), also when the run fails (status FAILED and the error): the size of the
# data, the seconds of each step (reading, forecasting, writing), the speed, the most memory used.
# Variables (scenario Load test, step 1): loadtest_clients_per_chunk (clients forecast at a time, like the project
# variable inference_clients_per_chunk), loadtest_model: dummy (default), or the name of a real forecaster
# (quantile_boosting, ...) to time it instead.
import json
import resource
import time

import dataiku
import pandas as pd

from cashflow import (FORECAST_COLUMNS, HORIZONS, Progress, append_row, history_days, iter_clients, period_matrix,
                      read_dataset, whole_number)
from forecast import forecast_table, one_row_per_client
from models import forecast, new_forecaster

LOG = {"run_at": "string", "status": "string", "error": "string", "model": "string", "cutoff_date": "string",
       "clients": "bigint", "months": "bigint", "postings_per_month": "bigint", "postings": "bigint",
       "daily_rows": "bigint", "parquet_files": "bigint", "parquet_mb": "double", "clients_per_chunk": "bigint",
       "chunks": "bigint", "clients_ok": "bigint", "clients_not_enough_history": "bigint",
       "seconds_generating": "double", "seconds_training_2_clients": "double", "seconds_reading": "double",
       "seconds_forecasting": "double", "seconds_writing": "double", "seconds_total": "double",
       "clients_per_second": "double", "daily_rows_per_second": "double", "peak_memory_mb": "bigint"}
started = time.time()
variables = dataiku.get_custom_variables()
log = {"run_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")}


class Dummy:
    """The dummy model: same inputs and outputs as a real forecaster (models.py), almost no computing. EV = the net
    cash flow of the last 30-day period, P10 / P90 = EV -/+ half of it."""
    name = "dummy"
    params = {}

    def fit(self, daily, clients, cutoff):
        period_matrix(daily, cutoff)              # the same data preparation as a real model
        return self

    def predict(self, daily, clients, cutoff):
        wide = period_matrix(daily, cutoff)
        if wide.empty:
            return pd.DataFrame(columns=FORECAST_COLUMNS)
        ev = wide.iloc[:, -1].fillna(0.0)
        one = pd.DataFrame({"p10": ev - ev.abs() / 2, "ev": ev, "p90": ev + ev.abs() / 2}).reset_index()
        return pd.concat([one.assign(window=h) for h in HORIZONS], ignore_index=True)[FORECAST_COLUMNS]


def step(what):
    print(f"PROGRESS load test 2/2, {what} ({time.time() - started:.0f} s so far)", flush=True)


def memory():
    return round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024)


def run():
    with dataiku.Folder("load_test_flows").get_download_stream("benchmark.json") as stream:
        info = json.loads(stream.read())
    log.update({k: info[k] for k in ("clients", "months", "postings_per_month", "postings", "daily_rows",
                                     "parquet_files", "parquet_mb", "seconds_generating")}, cutoff_date=info["cutoff"])
    cutoff = pd.Timestamp(info["cutoff"])
    name = variables.get("loadtest_model") or "dummy"
    make = Dummy if name == "dummy" else lambda: new_forecaster(name)
    per_chunk = whole_number("loadtest_clients_per_chunk", variables.get("loadtest_clients_per_chunk", 5000)) or 5000
    log.update(model=name, clients_per_chunk=per_chunk)

    step(f"fitting {name} on 2 clients only (no real training: only the forecast is timed)")
    ticked = time.time()
    clients = read_dataset("load_test_clients")
    first = next(iter_clients("load_test_flows", 2))
    model = make().fit(first, clients[clients["client_no"].isin(set(first["client_no"]))], cutoff)
    log["seconds_training_2_clients"] = round(time.time() - ticked, 1)
    del first

    forecasting, writing, chunks, counts, writer = 0.0, 0.0, 0, {}, None
    out, looping = dataiku.Dataset("load_test_forecast"), time.time()
    progress = Progress(f"load test 2/2, forecasting and writing load_test_forecast ({per_chunk:,} clients per "
                        "chunk)", len(clients))
    for daily in iter_clients("load_test_flows", per_chunk):
        ticked = time.time()
        group = clients[clients["client_no"].isin(set(daily["client_no"]))]
        table = forecast_table(forecast(model, daily, group, cutoff), history_days(daily, cutoff), cutoff)
        frame = one_row_per_client(table, cutoff).round(2)
        frame["model_id"], frame["forecaster"], frame["load_id"] = "load_test", name, "load_test"
        forecasting += time.time() - ticked
        ticked = time.time()
        if writer is None:
            out.write_schema_from_dataframe(frame)
            writer = out.get_writer()
        writer.write_dataframe(frame)
        writing += time.time() - ticked
        for status, n in frame["status"].value_counts().items():
            counts[status] = counts.get(status, 0) + int(n)
        chunks += 1
        progress.add(len(group))
    if writer is not None:
        writer.close()
    progress.finish()
    log.update(seconds_reading=round(time.time() - looping - forecasting - writing, 1),
               seconds_forecasting=round(forecasting, 1), seconds_writing=round(writing, 1), chunks=chunks,
               clients_ok=counts.get("OK", 0), clients_not_enough_history=counts.get("NOT_ENOUGH_HISTORY", 0))
    seconds = max(time.time() - looping, 0.1)                   # reading, forecasting and writing
    log.update(clients_per_second=round(sum(counts.values()) / seconds, 1),
               daily_rows_per_second=round(log["daily_rows"] / seconds, 1))


def write_log(status, error=""):
    log.update(status=status, error=error[:1000], seconds_total=round(time.time() - started, 1),
               peak_memory_mb=memory())
    append_row("load_test_log", LOG, log)


try:
    run()
except Exception as e:
    write_log("FAILED", f"{type(e).__name__}: {e}")
    raise
write_log("SUCCESS")
print("PROGRESS load test 2/2: done. Summary (also in load_test_log):", flush=True)
for key in ("model", "clients", "daily_rows", "parquet_mb", "chunks", "seconds_generating", "seconds_reading",
            "seconds_forecasting", "seconds_writing", "seconds_total",
            "clients_per_second", "peak_memory_mb"):
    print(f"PROGRESS    {key:<30} {log.get(key)}", flush=True)
'''

## 3. Build the flow

Uses the simulated warehouse tables, or links the warehouse's (read-only), and creates the Python recipes and their
CSV datasets, the folders `daily_flows` (Parquet files), `experiments` (tracked models) and `forecast_archive`, the
saved model `cash_forecast`, and the project variables `forecaster` (`quantile_boosting`),
`inference_clients_per_chunk` (5000) and the promotion rules (`promotion_*`) if they are not set yet, in 4 zones, and the Load test zone.
Training (folder `experiments`) and the promotion log are only rebuilt when asked by name, so the nightly batch never
retrains by accident.

In [ ]:
import json
import re
import sys
import time
import types

import pandas as pd

# Safe to run again in the same project: what already exists is kept (datasets and their data, folders, the saved
# model and its versions, zones, the project variables, forecasters.py), and the code of the Python recipes and of the
# rest of the library is updated.
flow = project.get_flow()


def load_module(name, code):
    """A library file as a module of this notebook: the code it writes to the project library."""
    module = types.ModuleType(name)
    sys.modules[name] = module
    exec(code, module.__dict__)
    return module


cashflow = load_module("cashflow", LIBRARY_CODE)
warehouse = load_module("warehouse", WAREHOUSE_CODE)
tracking = load_module("tracking", TRACKING_CODE)
RAW = [warehouse_table for warehouse_table, _ in warehouse.TABLES.values()]

# the input, checked before anything is built: the 2 warehouse tables, read-only through EDW_CONNECTION (SQL only)
try:
    database = client.get_connection(EDW_CONNECTION).get_info().get_type()      # Oracle, PostgreSQL, ...
except Exception as e:
    raise RuntimeError(f"Cannot use the connection {EDW_CONNECTION!r} ({e}). Check EDW_CONNECTION, or ask a Dataiku "
                       "admin for a read-only database connection to the warehouse, usable by your group.") from None
print(f"Input: {database} connection {EDW_CONNECTION}, read-only")
for table, (name, _) in warehouse.TABLES.items():
    columns = warehouse.read_columns(table)
    query = warehouse.table_query(table, f'"{EDW_SCHEMA}"."{name}"', end=pd.Timestamp.now())
    try:
        dataiku.SQLExecutor2(connection=EDW_CONNECTION).query_to_df(f"SELECT * FROM ({query}) q WHERE 1 = 0")
    except Exception as e:
        found = re.search(r"(ORA-\d+[^\\\n']*|ERROR: [^\\\n']*)", str(e))    # the database's own message
        reason = found.group(1).strip() if found else str(e).splitlines()[0][:200]
        raise RuntimeError(f"Cannot read {EDW_SCHEMA}.{name} through {EDW_CONNECTION} ({reason}). Check EDW_SCHEMA, "
                           f"that the table exists with the columns {', '.join(columns)}, and that the connection's "
                           "user may read it.") from None
    print(f"   {EDW_SCHEMA}.{name}: can be read ({len(columns)} columns used)")
print(f"Output: datasets on {OUTPUT_CONNECTION}, folders on {OUTPUT_FOLDER_CONNECTION}")
CSV = {"style": "excel", "charset": "utf8", "separator": ",", "quoteChar": '"', "escapeChar": "\\",
       "dateSerializationFormat": "ISO", "arrayMapFormat": "json", "parseHeaderRow": True, "compress": ""}
COMPRESSED = ["final_forecast", "load_test_forecast"]       # the big ones: compressed CSV (gz)

# Python recipes: (inputs, outputs, flow zone). Every dataset is a CSV file (compressed for the big one); daily_flows
# (Parquet files, read far faster than a dataset), experiments and forecast_archive are folders;
# cash_forecast is the saved model.
# Each zone has an owner, and the zones only meet through three interfaces:
#   1. the data: daily_flows and clients (Staging) -> Training and Inference (and the data scientist's lab notebook)
#   2. the forecaster (models.py, forecasters.py): the project variable forecaster names the one Retrain trains. The
#      data scientist tries forecasters first in the notebook forecast_lab, outside the flow, without touching
#      production.
#   3. the saved model's active version: training -> inference. Inference reads nothing else from training.
# The Load test zone is apart: fake data of the same shape and a dummy model, run by the scenario Load test; it never
# reads or writes the real data, models or logs.
DATA = ["daily_flows", "clients"]
FLOW = {
    "stage_tables": (RAW, warehouse.STAGED, "Staging"),
    "train_and_test": (DATA, ["experiments", "test_results"], "Training"),
    "champion_challenger": (DATA + ["experiments"], ["promotion_log"], "Training"),
    "final_forecast": (DATA + [cashflow.SAVED_MODEL], ["final_forecast", "forecast_archive", "inference_log"],
                       "Inference"),
    "load_test_data": ([], ["load_test_clients", "load_test_flows"], "Load test"),
    "load_test_run": (["load_test_clients", "load_test_flows"], ["load_test_forecast", "load_test_log"], "Load test"),
}
FOLDERS = ["daily_flows", "experiments", "forecast_archive", "load_test_flows"]
# logs: each run adds rows
APPENDED = ["staging_log", "promotion_log", "inference_log", "load_test_log"]
BUILT_BY_NAME = ["experiments", "promotion_log"]   # only built when asked by name

try:                                                # needs the project's admin right
    metadata = project.get_metadata()
    metadata["label"] = PROJECT_NAME
    project.set_metadata(metadata)
except Exception as e:
    if not any(w in str(e).lower() for w in ("unauthorized", "forbidden", "permission", "not allowed")):
        raise
    print(f"Warning: could not name the project {PROJECT_NAME!r} (no admin right on the project): name it by hand")

# the project library: shared (cashflow.py, warehouse.py), training and lab (models.py, tracking.py), inference
# (forecast.py). forecasters.py is the data scientist's: written only when it is created, never overwritten.
library = project.get_library()
python_folder = library.get_folder("python") or library.add_folder("python")
for file_name, code in (("cashflow.py", LIBRARY_CODE), ("warehouse.py", WAREHOUSE_CODE), ("models.py", MODELS_CODE),
                        ("tracking.py", TRACKING_CODE), ("forecast.py", FORECAST_CODE)):
    (python_folder.get_file(file_name) or python_folder.add_file(file_name)).write(code)
if not python_folder.get_file("forecasters.py"):
    python_folder.add_file("forecasters.py").write(FORECASTERS_CODE)
    print("forecasters.py created (the data scientist's forecasters)")

# project variables, kept if they are already set: forecaster (the one Retrain trains), inference_clients_per_chunk
# (how many clients final_forecast forecasts at a time)
variables = project.get_variables()
defaults = {"forecaster": "quantile_boosting",
            "inference_clients_per_chunk": 5000, **tracking.PROMOTION_RULES}
missing = {k: v for k, v in defaults.items() if variables["standard"].get(k) in (None, "")}
if missing:
    variables["standard"].update(missing)
    project.set_variables(variables)
# history_months, clients_limit: always the setup's settings (what staging reads: the first time and Full rebuild)
months = cashflow.whole_number("HISTORY_MONTHS", HISTORY_MONTHS)
limit = cashflow.whole_number("CLIENTS_LIMIT", CLIENTS_LIMIT)
variables["standard"].pop("history_years", None)                 # the older setting
variables["standard"].update(history_months="" if months is None else months, clients_limit="" if limit is None else limit)
project.set_variables(variables)
print("History read the first time (and by Full rebuild):",
      "the whole history" if months is None else f"the last {months} months up to the cut-off day",
      "| clients:", "all the business clients" if limit is None else f"the first {limit:,} business clients")
if months is not None and months < cashflow.MIN_TRAINING_MONTHS:
    print(f"WARNING: with {months} months of history, the first run only stages the data: training, promotion and "
          f"the forecast need at least {cashflow.MIN_TRAINING_MONTHS} months")
print("Forecasters: Retrain trains {forecaster}; inference forecasts "
      "{inference_clients_per_chunk} clients at a time".format(**variables["standard"]))
print("Promotion rules:", {k: variables["standard"][k] for k in tracking.PROMOTION_RULES})


# what the project already has (read once, kept up to date below)
datasets = {d["name"] for d in project.list_datasets()}
recipes = {r["name"] for r in project.list_recipes()}



def new_dataset(name):
    """Managed dataset of the project: a CSV file (comma-separated, with a header row) on OUTPUT_CONNECTION. Kept if it
    exists."""
    if name in datasets:
        return name
    datasets.add(name)
    dataset = project.new_managed_dataset(name).with_store_into(OUTPUT_CONNECTION).create()
    settings = dataset.get_settings()
    settings.get_raw()["formatType"] = "csv"
    settings.get_raw()["formatParams"].update(CSV, compress="gz" if name in COMPRESSED else "")
    if name in BUILT_BY_NAME:
        settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
    settings.save()
    return name


# the 2 warehouse tables: datasets linked read-only to the tables of EDW_SCHEMA through EDW_CONNECTION (nothing is
# created or written there). One already in the project is kept only if it is that table (a run of the setup again);
# anything else (fake data, CSV, an old project's table) stops the setup before it is used.
for name in RAW:
    if name in datasets:
        raw = project.get_dataset(name).get_settings().get_raw()
        params = raw.get("params", {})
        if (params.get("connection"), params.get("schema"), params.get("table")) != (EDW_CONNECTION, EDW_SCHEMA, name):
            raise RuntimeError(f"The project already has a dataset {name} that is not the table {EDW_SCHEMA}.{name} "
                               f"of {EDW_CONNECTION} (it is {raw.get('type')} on {params.get('connection')}): fake or "
                               f"old data. Delete it (Flow > {name} > Delete), or use a new empty project, then run "
                               "the setup again.")
        print(f"{name}: already linked to {EDW_SCHEMA}.{name} of {EDW_CONNECTION}")
        continue
    datasets.add(name)
    dataset = project.create_sql_table_dataset(name, database, EDW_CONNECTION, name, EDW_SCHEMA)
    dataset.autodetect_settings().save()           # the table's columns, read from the database
    if not dataset.get_settings().get_raw().get("schema", {}).get("columns"):
        # staging reads with its own SELECT, so it still works; only Dataiku's view of the table would fail
        print(f"WARNING: Dataiku could not read the columns of {EDW_SCHEMA}.{name}: opening the dataset in Dataiku "
              "will fail. Fix: the dataset's Settings > Test table & get schema > Save.")
    print(f"{name}: linked to {EDW_SCHEMA}.{name} of {EDW_CONNECTION}")
override = variables["standard"].get("cutoff_override")
if override:
    print(f"WARNING: the project variable cutoff_override is {override}: the forecast uses that day instead of "
          "yesterday. Delete it (Project > Variables) unless you mean it.")

existing_folders = {f["name"]: f["id"] for f in project.list_managed_folders()}
folders = {}
for name in FOLDERS:
    if name in existing_folders:
        folders[name] = project.get_managed_folder(existing_folders[name])
        continue
    folders[name] = project.create_managed_folder(name, connection_name=OUTPUT_FOLDER_CONNECTION)
    if name in BUILT_BY_NAME:
        folder_settings = folders[name].get_settings()
        folder_settings.get_raw()["flowOptions"]["rebuildBehavior"] = "EXPLICIT"
        folder_settings.save()
existing_models = {m["name"]: m["id"] for m in project.list_saved_models()}
saved_model = (project.get_saved_model(existing_models[cashflow.SAVED_MODEL]) if cashflow.SAVED_MODEL in existing_models
               else project.create_mlflow_pyfunc_model(cashflow.SAVED_MODEL, "REGRESSION"))
ref = lambda name: (folders[name].id if name in folders else saved_model.id if name == cashflow.SAVED_MODEL
                    else name)

for name, (inputs, outputs, _) in FLOW.items():
    for item in inputs + outputs:
        if item not in folders and item != cashflow.SAVED_MODEL:
            new_dataset(item)
    if name in recipes:                # already there: its inputs and outputs changed? then made again
        settings = project.get_recipe(name).get_settings()
        if (set(settings.get_flat_input_refs()) != set(map(ref, inputs))
                or set(settings.get_flat_output_refs()) != set(map(ref, outputs))):
            project.get_recipe(name).delete()
            recipes.discard(name)
            print("recipe", name, "has new inputs or outputs: made again")
    if name in recipes:                # already there: only its code and code env are updated
        recipe = project.get_recipe(name)
    else:
        builder = project.new_recipe("python", name)
        for i in inputs:
            builder.with_input(ref(i))
        for o in outputs:
            builder.with_output(ref(o), append=o in APPENDED)
        recipe = builder.create()
        recipes.add(name)
    settings = recipe.get_settings()
    settings.set_payload(RECIPES[name])
    settings.set_code_env(code_env=CODE_ENV)
    settings.save()
    print("python recipe", name, "->", ", ".join(outputs))

ZONES = {"Raw data": RAW, "Staging": list(warehouse.STAGED), "Training": [], "Inference": [], "Load test": []}
for _, outputs, zone_name in FLOW.values():
    ZONES[zone_name] += [o for o in outputs if o not in ZONES[zone_name]]
# the raw data goes in Dataiku's Default zone, renamed (so there is no empty zone). It exists once another zone does.
existing_zones = {z.name: z for z in flow.list_zones()}
zones = {z: existing_zones.get(z) or flow.create_zone(z) for z in ZONES if z != "Raw data"}
zones["Raw data"] = existing_zones.get("Raw data") or flow.get_default_zone()
if zones["Raw data"].name != "Raw data":
    zone_settings = zones["Raw data"].get_settings()
    zone_settings.name = "Raw data"
    zone_settings.save()


def safe_add(zone, item):
    """Put an item in a zone (already there: nothing to do)."""
    try:
        zone.add_item(item)
    except Exception as e:
        if "already" not in str(e).lower():
            raise


for zone_name, items in ZONES.items():
    for item in items:
        safe_add(zones[zone_name], folders[item] if item in folders else project.get_dataset(item))
safe_add(zones["Inference"], saved_model)    # next to the recipe that uses it
print("Flow ready:", ", ".join(zones))

## 4. First run

Builds the daily flows (the whole history, the first time), trains, tests and deploys a model, runs champion / challenger (with no production model yet,
the first model is promoted, with a warning in promotion_log if its checks fail), then runs the nightly batch:
forecast, archive, log.

In [ ]:
def build(step, items, mode):
    """Runs a Dataiku job and waits for it, showing its PROGRESS lines here (also in the job's log) every 30 s."""
    print(f"\n{step} ...", flush=True)
    builder = project.new_job(mode).with_auto_update_schema_before_each_recipe_run(True)
    for item in items:
        if item in folders:
            builder.with_output(folders[item].id, object_type="MANAGED_FOLDER")
        else:
            builder.with_output(item)
    job, shown, began = builder.start(), 0, time.time()
    while True:
        state = job.get_status().get("baseStatus", {}).get("state", "")
        try:
            lines = [l[l.index("PROGRESS ") + 9:] for l in job.get_log().splitlines() if "PROGRESS " in l]
        except Exception:
            lines = []
        for line in lines[shown:]:
            print("   ", line, flush=True)
        shown = max(shown, len(lines))
        if state in ("DONE", "FAILED", "ABORTED"):
            break
        time.sleep(30 if time.time() - began > 60 else 5)
    if state != "DONE":
        raise RuntimeError(f"{step}: the job {state}. Open it in Jobs (id {job.id}) to read its log.")
    print(f"{step}: done in {(time.time() - began) / 60:.1f} min", flush=True)


started = time.time()
warehouse.set_cutoff()                                          # the first 2 steps of every scenario
warehouse.check_warehouse_loaded()
build("1/4 Staging: read the warehouse (clients and daily flows)", ["daily_flows"], "RECURSIVE_FORCED_BUILD")
staged_only = months is not None and months < cashflow.MIN_TRAINING_MONTHS
if staged_only:                                                # too little history to train: stop after staging
    print(f"\nStaging done in {(time.time() - started) / 60:.1f} minutes. Steps 2-4 (training, promotion, first "
          f"forecast) are skipped: they need at least {cashflow.MIN_TRAINING_MONTHS} months of history, and "
          f"HISTORY_MONTHS is {months}.")
else:
    build("2/4 Train, test, track and deploy the first model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD")
    build("3/4 Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD")
    production = tracking.apply_promotion()                    # activates its version of cash_forecast
    build("4/4 Forecast every active client, archive and log the run", ["inference_log"], "RECURSIVE_FORCED_BUILD")
    print(f"First run done in {(time.time() - started) / 60:.1f} minutes: production model {production}")

## 5. Scenarios

- **Nightly batch**: set the cut-off, add the new days to the daily flows, forecast with the production model,
  archive it and log the run. It never trains.
- **Retrain, test and promote**: set the cut-off, add the new days to the daily flows, train, test and deploy a new model with
  the forecaster the project variable `forecaster` names, champion / challenger with the promotion rules, activate
  it if it passes. It does not forecast: the next Nightly batch uses the new active version.

- **Full rebuild of the daily flows** (run by hand, no trigger): empty `daily_flows` and `staging_log`, then read the
  whole history again. For example after a reversal cancels a posting from a day already staged.
- **Load test** (run by hand, no trigger): set the numbers in its step 1, run it, read `load_test_log`.

Their triggers (daily at 06:00 and monthly on the 5th, Riyadh time) are created switched off; switch them on
in production.

In [ ]:
# Production runs: two independent scenarios. Nightly batch (inference) forecasts with the active version of the saved
# models; Retrain, test and promote (training) adds a version and activates it if it is better, and never forecasts.
# Each starts by setting the cut-off day (yesterday) and checking the warehouse load of that day is finished (Python
# steps; the check is a mock to replace with the bank's); the staging recipe adds the new days to daily_flows and
# stops if the postings do not reach the cut-off day. A third one, Full rebuild of the daily flows, has no trigger:
# run it by hand to read the whole history again. The tracked models (folder experiments) and the promotion log are only rebuilt when built by name, so the
# nightly batch never retrains by accident.
def step(n, label, what, mode=None):
    common = {"id": f"step_{n}", "name": label, "enabled": True, "runConditionType": "RUN_IF_STATUS_MATCH",
              "runConditionStatuses": ["SUCCESS", "WARNING"]}
    if isinstance(what, dict):          # variables for the next steps, edited on the scenario's page
        return {**common, "type": "define_vars", "params": {"variables": what, "definitions": []}}
    if isinstance(what, str):           # a Python step: library function to call
        module = "warehouse" if what in ("set_cutoff", "check_warehouse_loaded", "clear_daily_flows") else "tracking"
        return {**common, "type": "custom_python", "params": {
            "script": f"from {module} import {what}\n\n{what}()\n", "envSelection": {"envMode": "INHERIT"},
            "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}
    kind = lambda i: ("MANAGED_FOLDER", folders[i].id) if i in folders else ("DATASET", i)
    builds = [{"type": t, "itemId": item, "partitionsSpec": ""} for t, item in map(kind, what)]
    return {**common, "type": "build_flowitem", "params": {
        "builds": builds, "jobType": mode, "proceedOnFailure": False, "handleWarningsAs": "WARNING"}}


# the load test's numbers, in step 1 of the scenario Load test (kept when the setup runs again):
#   loadtest_clients: fake clients; loadtest_months: months of history; loadtest_postings_per_month: average postings
#   per client and month; loadtest_clients_per_chunk: clients forecast at a time; loadtest_model: dummy, or a
#   forecaster's name (quantile_boosting, ...) to time a real model
LOAD_TEST = {"loadtest_clients": 1000, "loadtest_months": 24, "loadtest_postings_per_month": 40,
             "loadtest_clients_per_chunk": 5000, "loadtest_model": "dummy"}
START = [("Set the cut-off day (yesterday)", "set_cutoff"),
         ("Check the warehouse load is finished", "check_warehouse_loaded")]
NIGHTLY = [
    *START,
    ("Forecast with the production model, archive and log it", ["inference_log"], "RECURSIVE_FORCED_BUILD"),
]
SCENARIOS = {
    "Nightly batch": ("After each warehouse load: forecast the next 30, 60 and 90 days with the production model (the active "
                      "version of the saved models), archive it and log the run", "daily", NIGHTLY),
    "Retrain, test and promote": (
        "Train and test a new model with the forecaster the project variable forecaster names (tracked in Experiment "
        "Tracking, deployed as a saved model version), and activate it if it beats the production model. It does not "
        "forecast: the Nightly batch uses the active version", "monthly", [
            *START,
            ("Prepare the data (daily flows, clients)", ["daily_flows"], "RECURSIVE_FORCED_BUILD"),
            ("Train, test and deploy a new model", ["experiments"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Champion / challenger", ["promotion_log"], "NON_RECURSIVE_FORCED_BUILD"),
            ("Promote if better", "apply_promotion"),
        ]),
    "Full rebuild of the daily flows": (
        "Run by hand: empties daily_flows and staging_log, then reads the whole history from the warehouse again "
        "(the nightly runs only read the new days). For example after a reversal cancels a posting of a day already "
        "staged. It does not forecast: run the Nightly batch after it", None, [
            *START,
            ("Empty the daily flows", "clear_daily_flows"),
            ("Read the whole history again", ["daily_flows"], "RECURSIVE_FORCED_BUILD"),
        ]),
    "Load test": (
        "Run by hand: times the nightly forecast (read in chunks, forecast, write; no real training: the model is "
        "fitted on 2 clients) on fake data with a dummy model, in the Load test zone only (it never touches the real data, models or logs). Set the numbers "
        "in step 1 (Steps > Set the load test numbers), run it, then read load_test_log (one row per run)", None, [
            ("Set the load test numbers", LOAD_TEST),
            ("Fake data, forecast, log", ["load_test_log"], "RECURSIVE_FORCED_BUILD"),
        ]),
}
existing_scenarios = {s["name"]: s["id"] for s in project.list_scenarios()}
for name, (description, schedule, steps) in SCENARIOS.items():
    if name in existing_scenarios:     # run again: update its steps, keep its triggers and run history
        settings = project.get_scenario(existing_scenarios[name]).get_settings()
        settings.get_raw()["description"] = description
        before = {s["id"]: s for s in settings.get_raw()["params"].get("steps", [])}
        new_steps = [step(n, *s) for n, s in enumerate(steps)]
        for s in new_steps:              # numbers already set on the scenario's page are kept
            if s["type"] == "define_vars" and before.get(s["id"], {}).get("type") == "define_vars":
                s["params"] = before[s["id"]]["params"]
        settings.get_raw()["params"]["steps"] = new_steps
        settings.save()
        print(f"Scenario updated: {name}")
        continue
    scenario = project.create_scenario(name, "step_based", {
        "active": False, "description": description,
        "params": {"steps": [step(n, *s) for n, s in enumerate(steps)]}})
    if schedule is None:                           # run by hand only
        print(f"Scenario ready: {name} (no trigger: run it by hand)")
        continue
    settings = scenario.get_settings()
    if schedule == "daily":
        settings.add_daily_trigger(hour=6, timezone="Asia/Riyadh")
    else:
        settings.add_monthly_trigger(day=5, hour=7, timezone="Asia/Riyadh")
    settings.raw_triggers[-1]["active"] = False    # switch on in production (Automation node)
    try:
        settings.save()
        note = "trigger added, switched off"
    except Exception as e:                         # licences without time-based triggers (Free Edition)
        if "License" not in str(e):
            raise
        note = "no trigger (not in this licence): add it on the Automation node"
    print(f"Scenario ready: {name} ({note})")

## 6. Results

To use another model (for example to roll back), activate its version in the saved model `cash_forecast` and run
the Nightly batch. To change when a model is promoted, change the project variables `promotion_*` (no code change).

In [ ]:
if staged_only:                                                 # only staging ran (HISTORY_MONTHS too short)
    clients_staged = dataiku.Dataset("clients").get_dataframe()
    print(f"Staged: {len(clients_staged):,} business clients, {sum(len(p) for p in cashflow.iter_daily_parts()):,} "
          "daily_flows rows (per client, day and category)")
    print("Staging log (one row per staging run: what was read from the warehouse):")
    display(dataiku.Dataset("staging_log").get_dataframe().sort_values("run_at"))
else:
    print("Production model (active version of the saved model cash_forecast):", tracking.production_model_id())
    print("Promotion log:")
    display(dataiku.Dataset("promotion_log").get_dataframe())
    print("Test on the last 6 periods of 30 days: TOTAL rows (net per client), category rows, and ALL together, by window (1 = days 1-30, 2 = 31-60, 3 = 61-90):")
    display(dataiku.Dataset("test_results").get_dataframe()[
        ["level", "window", "rows", "wape_model", "wape_baseline", "coverage_p10_p90", "below_p10", "above_p90"]])
    final = dataiku.Dataset("final_forecast").get_dataframe()
    print(f"Final forecast: one row per active client, {len(final)} rows, {final.shape[1]} columns. Status:",
          final["status"].value_counts().to_dict())
    one = final[final["status"] == "OK"].iloc[0]
    print(f"One client, {one['client_no']} (TOTAL and the first categories; empty: a category it does not have):")
    display(one.dropna().head(40).to_frame("value"))
    print("Inference log (one row per forecast run):")
    display(dataiku.Dataset("inference_log").get_dataframe().sort_values("run_at"))
    print("Staging log (one row per staging run: what was read from the warehouse):")
    display(dataiku.Dataset("staging_log").get_dataframe().sort_values("run_at"))